# Nesine – bugünün ve gece 03:00'e kadarki maçların bütün oranları, isimleriyle
**Her gün: menüden Çalışma zamanı → Tümünü çalıştır (Ctrl+F9).** İki Excel iner: başlamamış maçlar (4. hücre) ve şu an canlı oynanan maçlar (6. hücre).

Bültende market ve seçenek adları yok, sadece kodlar var. Not defteri, 17.11.2026 gerçek bülten verisiyle doğrulanmış **104 futbol market kodunu** hazır içeriyor; her gün ek bir şey yapmadan kullanılabilir.

Yeni bir spor (basketbol, tenis…) ya da sözlükte olmayan marketler için 3. hücredeki `REFERANSLAR` listesine sitedeki maç sayfasından kopyaladığın metni ekle. Metni kopyaladıktan **hemen sonra** çalıştır; birden çok örnek varsa birlikte değerlendirilir. Öğrenilenler `market_sozlugu_v3.json` dosyasına kaydedilir.

In [ ]:
# 1) Fonksiyonlar
import requests, json, pandas as pd
from datetime import datetime

URL = "https://cdnbulten.nesine.com/api/bulten/getprebultenfull"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "tr-TR,tr;q=0.9",
    "Origin": "https://www.nesine.com",
    "Referer": "https://www.nesine.com/",
}

def bulteni_cek():
    r = requests.get(URL, headers=HEADERS, timeout=30)
    print("HTTP durumu:", r.status_code, "| boyut:", len(r.content) // 1024, "KB")
    r.raise_for_status()
    return r.json()

def mac_listesini_bul(veri):
    """Beklenen yer veri['sg']['EA']. Yapı değişmişse, içinde 'MA' (marketler)
    olan en büyük sözlük listesini arayıp onu döndürür."""
    try:
        ea = veri["sg"]["EA"]
        if isinstance(ea, list) and ea:
            return ea
    except (KeyError, TypeError):
        pass
    en_iyi = []
    def gez(x):
        nonlocal en_iyi
        if isinstance(x, list):
            if x and all(isinstance(i, dict) for i in x[:5]) and any("MA" in i for i in x[:5]):
                if len(x) > len(en_iyi):
                    en_iyi = x
            for i in x:
                gez(i)
        elif isinstance(x, dict):
            for v in x.values():
                gez(v)
    gez(veri)
    return en_iyi

def duzlestir(maclar):
    """Her satır = bir maçın bir marketindeki bir seçenek."""
    satirlar = []
    for m in maclar:
        temel = {
            "kod": m.get("C"),
            "spor_kodu": m.get("TYPE"),
            "lig": m.get("LN"),
            "tarih": m.get("D"),
            "saat": m.get("T"),
            "ev": m.get("HN"),
            "deplasman": m.get("AN"),
            "karsilasma": f"{m.get('HN')} - {m.get('AN')}",
            "mac_mbs": m.get("MBS"),
        }
        for market in m.get("MA") or []:
            for sec in market.get("OCA") or []:
                satirlar.append({
                    **temel,
                    "market_id": market.get("MTID"),
                    "market_mbs": market.get("MBS"),
                    "ozel_deger": market.get("SOV"),   # ör. Alt/Üst için 2.5
                    "secenek_no": sec.get("N"),
                    # kapalı seçenekler bültende 1.0 geliyor -> boş bırak (sitedeki '-')
                    "oran": (lambda v: v if v > 1.0 else float("nan"))(pd.to_numeric(sec.get("O"), errors="coerce")),
                    # JSON seçenekte isim taşıyan bir alan varsa (oyuncu adı vb.) yedek etiket olarak tut
                    "json_etiket": next((v for k, v in sec.items() if k not in ("N", "O")
                                         and isinstance(v, str) and v.strip()), None),
                })
    return pd.DataFrame(satirlar)

def mac_sonucu_tablosu(df, ms_market_id=1):
    """Maç Sonucu (1-X-2) marketini geniş tabloya çevirir.
    ms_market_id varsayımı 1'dir; çıktıda yanlış görünürse değiştir."""
    ms = df[df["market_id"] == ms_market_id]
    if ms.empty:
        return ms
    genis = ms.pivot_table(index=["kod", "lig", "tarih", "saat", "ev", "deplasman"],
                           columns="secenek_no", values="oran", aggfunc="first").reset_index()
    genis = genis.rename(columns={1: "MS1", 2: "MS0", 3: "MS2"})
    genis.columns.name = None
    return genis


import re
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
TR = ZoneInfo("Europe/Istanbul")

def mac_zamani(m):
    """Maçın başlama zamanını Türkiye saatiyle döndürür. Birkaç olası formatı dener."""
    d, t = m.get("D"), m.get("T")
    try:
        if isinstance(d, str) and re.fullmatch(r"\d{2}\.\d{2}\.\d{4}", d):
            return datetime.strptime(f"{d} {t or '00:00'}", "%d.%m.%Y %H:%M").replace(tzinfo=TR)
        if isinstance(d, str) and re.match(r"\d{4}-\d{2}-\d{2}", d):
            dt = datetime.fromisoformat(d.replace("Z", "+00:00"))
            if dt.tzinfo is None:
                dt = dt.replace(tzinfo=TR)
            if t and dt.hour == 0 and dt.minute == 0 and re.fullmatch(r"\d{1,2}:\d{2}", str(t)):
                h, mi = map(int, str(t).split(":")); dt = dt.replace(hour=h, minute=mi)
            return dt.astimezone(TR)
    except Exception:
        pass
    for k in ("ESD", "SD", "DT"):   # olası epoch alanları
        v = m.get(k)
        if isinstance(v, (int, float)) and v > 1e9:
            return datetime.fromtimestamp(v / 1000 if v > 1e12 else v, tz=TR)
    return None

def zaman_penceresi(simdi=None, bitis_saati=3):
    """Bugün 00:00'dan, bir sonraki sabah 03:00'e kadar (TR saati).
    Gece 00:00-03:00 arasında çalıştırırsan dünkü günü baz alır."""
    simdi = simdi or datetime.now(TR)
    gun = simdi.date() if simdi.hour >= bitis_saati else (simdi - timedelta(days=1)).date()
    bas = datetime(gun.year, gun.month, gun.day, tzinfo=TR)
    bit = bas + timedelta(days=1, hours=bitis_saati)
    return bas, bit

def pencereye_gore_filtrele(maclar, bas, bit):
    secilen, zamansiz = [], 0
    for m in maclar:
        z = mac_zamani(m)
        if z is None:
            zamansiz += 1
            continue
        if bas <= z < bit:
            secilen.append(m)
    if zamansiz:
        print(f"Uyarı: {zamansiz} maçın saati okunamadı (alan formatı farklı olabilir).")
    return secilen

def genis_tablo(df):
    """Her maç tek satır; her market/seçenek ayrı sütun ('Maç Sonucu | MS 1' gibi)."""
    d = df.copy()
    if "market_adi" in d:
        d["sutun"] = d["market_adi"] + " | " + d["secenek_adi"]
    else:
        d["sutun"] = "m" + d["market_id"].astype(str) + "_" + d["ozel_deger"].astype(str) + "_s" + d["secenek_no"].astype(str)
    sira = d.sort_values(["market_id", "ozel_deger", "secenek_no"])["sutun"].drop_duplicates().tolist()
    idx = [c for c in ["kod", "karsilasma", "spor_kodu", "lig", "tarih", "saat", "ev", "deplasman"] if c in d]
    d[idx] = d[idx].fillna("")
    g = d.pivot_table(index=idx, columns="sutun", values="oran", aggfunc="first")
    g = g[[c for c in sira if c in g.columns]].reset_index()
    g.columns.name = None
    return g

def mac_goster(df, kod=None, takim=None):
    """Tek maçın bütün bahislerini market market yazdırır."""
    s = df
    if kod is not None: s = s[s["kod"] == kod]
    if takim: s = s[s["ev"].fillna("").str.contains(takim, case=False) | s["deplasman"].fillna("").str.contains(takim, case=False)]
    if s.empty:
        print("Maç bulunamadı."); return
    for (k, ev, dep), mac in s.groupby(["kod", "ev", "deplasman"]):
        print(f"\n=== {ev} - {dep} | {mac['tarih'].iloc[0]} {mac['saat'].iloc[0]} | kod {k} | {mac['market_id'].nunique()} market")
        mac = mac.sort_values(["market_id", "ozel_deger", "secenek_no"])
        for ad, mk in mac.groupby("market_adi", sort=False):
            print(f"  {ad}:  " + "   ".join(f"{e} {o}" for e, o in zip(mk["secenek_adi"], mk["oran"])))


import io, contextlib
import re, json
try:
    from scipy.optimize import linear_sum_assignment
except ImportError:
    linear_sum_assignment = None

def _oran_mi(s):
    """Oran satırı mı? '1.64', '23.00', '130', '-' evet; '1', '3', '0' (sayaç/etiket) hayır."""
    if s == "-":
        return True
    if re.fullmatch(r"\d+\.\d+", s):
        return True
    return re.fullmatch(r"\d+", s) is not None and int(s) >= 10

def _f(x):
    """Bültende kapalı seçenekler 1.0 olarak geliyor; onları 'kapalı' (None) say."""
    try:
        v = float(x)
    except (TypeError, ValueError):
        return None
    return v if v > 1.0 else None

def _sayi(s):
    return None if s == "-" else float(s)

def referans_ayristir(metin):
    """Nesine maç sayfasından kopyalanan metni [{ad, secenekler:[(etiket, oran|None)]}] listesine çevirir."""
    sat = [s.strip() for s in metin.splitlines() if s.strip()]
    marketler, mevcut, i = [], None, 0
    while i < len(sat):
        s = sat[i]
        if mevcut is not None and i + 1 < len(sat) and not _oran_mi(s) and _oran_mi(sat[i + 1]):
            mevcut["secenekler"].append((s, _sayi(sat[i + 1])))
            i += 2
            continue
        if i + 2 < len(sat) and not _oran_mi(s) and not _oran_mi(sat[i + 1]) and _oran_mi(sat[i + 2]):
            mevcut = {"ad": s, "secenekler": []}
            marketler.append(mevcut)
        else:
            mevcut = None
        i += 1
    return [m for m in marketler if m["secenekler"]]

def _cizgi_uyumlu(ad, sov):
    """Market adında çizgi (2,5 / 1:0 gibi) varsa JSON'daki çizgiyle çelişmesin."""
    if sov in (None, 0) or sov != sov:
        return True
    if not re.search(r"\d", ad):        # adında çizgi yoksa (İlk Korner gibi) kontrol etme
        return True
    yazim = f"{abs(sov):g}"
    return yazim.replace(".", ",") in ad or yazim in ad

def _eslestir(ref_sec, json_sec):
    """Referans seçenekleri ile JSON seçeneklerini oran değerine göre eşleştirir.
    Dönüş: (ortalama göreli fark, {N: etiket}) ya da None."""
    mevcut = [(j, e, o) for j, (e, o) in enumerate(ref_sec) if o is not None]
    js = [(n, o) for n, o in json_sec if o]
    if not mevcut or len(js) < len(mevcut) or len(js) > len(ref_sec):
        return None
    n_ref, n_js = len(ref_sec), len(js)
    maliyet = [[abs(o - js[c][1]) / o + 0.002 * abs(j / max(n_ref - 1, 1) - c / max(n_js - 1, 1))
                for c in range(n_js)] for (j, e, o) in mevcut]
    if linear_sum_assignment is not None:
        satir, sutun = linear_sum_assignment(maliyet)
        cift = list(zip(satir, sutun))
    else:   # scipy yoksa açgözlü eşleştirme
        tum = sorted((maliyet[r][c], r, c) for r in range(len(mevcut)) for c in range(n_js))
        kr, kc, cift = set(), set(), []
        for _, r, c in tum:
            if r not in kr and c not in kc:
                kr.add(r); kc.add(c); cift.append((r, c))
    fark = sum(abs(mevcut[r][2] - js[c][1]) / mevcut[r][2] for r, c in cift) / len(mevcut)
    etiketler = {js[c][0]: mevcut[r][1] for r, c in cift}
    # kapalı ('-') seçenekler: kalan JSON seçeneklerine sırayla
    eksik = [e for e, o in ref_sec if o is None]
    kalan = [n for n, _ in json_sec if n not in etiketler]
    for n, e in zip(kalan, eksik):
        etiketler[n] = e
    return fark, etiketler

def _sirali_fark(ref_sec, json_sec):
    """Seçenek sırası sitedekiyle aynı varsayımıyla: aynı sıradaki oranları karşılaştırır.
    Kapalı/açık durumu tutmayan her pozisyon için ceza ekler."""
    if len(ref_sec) != len(json_sec):
        return None
    farklar, uyumsuz = [], 0
    for (e, ro), (n, jo) in zip(ref_sec, json_sec):
        if ro is not None and jo is not None:
            farklar.append(abs(ro - jo) / ro)
        elif (ro is None) != (jo is None):
            uyumsuz += 1
    if not farklar:
        return None
    return sum(farklar) / len(farklar) + 0.05 * uyumsuz, len(farklar)

def _sablon(ad, sov):
    """'2,5 Gol Alt/Üst' + çizgi 2.5 -> '{cizgi} Gol Alt/Üst' (başka çizgilere de uysun diye)."""
    if sov in (None, 0) or sov != sov:
        return ad
    yazim = f"{sov:g}".replace(".", ",")
    return ad.replace(yazim, "{cizgi}", 1) if yazim in ad else ad

OZEL_KELIMELER = ("Oyuncu", "Kaleci", "Karşılaşma", "Takım ")   # seçenekleri maçtan maça değişen marketler

def sozluk_ogren(mac, referans, esik=0.06):
    """Bir maçın JSON'u + aynı maçın sitedeki metni -> market sözlüğü.
    1. tur: seçenek sırası sitedekiyle aynı kabul edilerek sıralı karşılaştırma (normal marketler).
    2. tur: eşleşmeyenler için sıradan bağımsız, oran değerine göre eşleştirme (oyuncu listeleri vb.).
    esik: kabul edilen ortalama göreli oran farkı."""
    ev, dep = str(mac.get("HN", "")), str(mac.get("AN", ""))
    ma = mac.get("MA") or []
    js_list = [sorted([(o.get("N"), _f(o.get("O"))) for o in mk.get("OCA") or []],
                      key=lambda x: (x[0] is None, x[0])) for mk in ma]
    sozluk, rapor = {}, []
    kullanilan_r, kullanilan_m = set(), set()

    def kaydet(ri, mi, fark, etiketler, yontem, belirsiz=False):
        mk, rm = ma[mi], referans[ri]
        kullanilan_r.add(ri); kullanilan_m.add(mi)
        mtid, sov = mk.get("MTID"), mk.get("SOV")
        tek_oran = sum(o is not None for _, o in rm["secenekler"]) == 1
        sec_map = {str(n): sablon_etiket(e, ev, dep) for n, e in etiketler.items()}
        maca_ozel = rm["ad"].startswith(OZEL_KELIMELER)
        bilgi = {"fark": round(fark, 4), "supheli": bool(tek_oran or belirsiz),
                 "maca_ozel": maca_ozel, "ogrenilen_mac": mac.get("C"), "yontem": yontem}
        sozluk[f"{mtid}|{sov}"] = {"ad": rm["ad"], "secenekler": sec_map, **bilgi}
        sozluk.setdefault(f"{mtid}|*", {"ad": _sablon(rm["ad"], sov), "secenekler": dict(sec_map), **bilgi})
        not_ = "; ".join(x for x in ["tek oranla eşleşti" if tek_oran else "",
                                      "benzer oranlı başka market var" if belirsiz else "",
                                      "oyuncu adları maça özel" if maca_ozel else ""] if x)
        rapor.append({"market": rm["ad"], "market_id": mtid, "cizgi": sov, "secenek": len(rm["secenekler"]),
                      "oran_farki_%": round(fark * 100, 1), "yontem": yontem, "not": not_})

    # --- 1. tur: sıralı ---
    adaylar = []
    for ri, rm in enumerate(referans):
        for mi, mk in enumerate(ma):
            if not _cizgi_uyumlu(rm["ad"], mk.get("SOV")):
                continue
            r = _sirali_fark(rm["secenekler"], js_list[mi])
            if r and r[0] <= esik:
                adaylar.append((r[0], -r[1], ri, mi))
    adaylar.sort()
    en_iyi = {}
    for fark, _, ri, mi in adaylar:
        en_iyi.setdefault(ri, []).append(fark)
    for fark, _, ri, mi in adaylar:
        if ri in kullanilan_r or mi in kullanilan_m:
            continue
        f = en_iyi[ri]
        belirsiz = len(f) > 1 and (f[1] - f[0]) < 0.005
        etiketler = {n: e for (e, _), (n, _) in zip(referans[ri]["secenekler"], js_list[mi])}
        kaydet(ri, mi, fark, etiketler, "sıralı", belirsiz)

    # --- 2. tur: sıradan bağımsız (oyuncu listeleri gibi) ---
    adaylar = []
    for ri, rm in enumerate(referans):
        if ri in kullanilan_r:
            continue
        for mi, mk in enumerate(ma):
            if mi in kullanilan_m or not _cizgi_uyumlu(rm["ad"], mk.get("SOV")):
                continue
            if len(js_list[mi]) != len(rm["secenekler"]):
                continue
            r = _eslestir(rm["secenekler"], js_list[mi])
            if r and r[0] <= esik:
                adaylar.append((r[0], ri, mi, r[1]))
    adaylar.sort(key=lambda x: x[0])
    for fark, ri, mi, etiketler in adaylar:
        if ri in kullanilan_r or mi in kullanilan_m:
            continue
        kaydet(ri, mi, fark, etiketler, "değere göre")

    eslesmeyen = [rm["ad"] for ri, rm in enumerate(referans) if ri not in kullanilan_r]
    print(f"   sıralı eşleşen: {sum(r['yontem']=='sıralı' for r in rapor)}, değere göre eşleşen: "
          f"{sum(r['yontem']!='sıralı' for r in rapor)}, eşleşmeyen: {len(eslesmeyen)}")
    return sozluk, rapor, eslesmeyen

def sablon_etiket(e, ev, dep):
    if ev and ev in e: e = e.replace(ev, "{ev}")
    if dep and dep in e: e = e.replace(dep, "{dep}")
    return e

def _agirlik(v):
    """Bir eşleşmenin oy ağırlığı: şüpheli (benzer oranlı/tek oranlı) yarım, oran farkı büyükse daha az."""
    w = 0.5 if v.get("supheli") else 1.0
    return w * (1.0 if v.get("fark", 1) < 0.01 else 0.6)

def sozluk_birlestir(eski, yeni):
    """Aynı market kodu için farklı örneklerin verdiği isimler oylanır; en yüksek toplam ağırlıklı isim kazanır.
    Oylar dosyada saklanır, yeni örnekler eklendikçe sözlük kendini düzeltir."""
    son = dict(eski)
    for k, v in yeni.items():
        v = {a: b for a, b in v.items() if a != "adaylar"}
        e = son.get(k)
        adaylar = dict(e.get("adaylar", {})) if e else {}
        if e and not adaylar:          # eski formatta kayıt
            adaylar[e["ad"]] = {"oy": _agirlik(e), "kayit": {a: b for a, b in e.items() if a != "adaylar"}}
        a = adaylar.setdefault(v["ad"], {"oy": 0.0, "kayit": v})
        a["oy"] = round(a["oy"] + _agirlik(v), 3)
        if (v.get("supheli"), v.get("fark", 1)) < (a["kayit"].get("supheli"), a["kayit"].get("fark", 1)):
            a["kayit"] = v
        kazanan_ad = max(adaylar, key=lambda n: adaylar[n]["oy"])
        if e and e["ad"] != kazanan_ad and not k.endswith("|*"):
            print(f"   ! Kod {k}: '{e['ad']}' yerine oylarla '{kazanan_ad}' seçildi " +
                  str({n: x["oy"] for n, x in adaylar.items()}))
        elif e and len(adaylar) > 1 and v["ad"] != kazanan_ad and not k.endswith("|*"):
            print(f"   ! Kod {k}: bu örnek '{v['ad']}' dedi ama oylarla '{kazanan_ad}' kaldı " +
                  str({n: x["oy"] for n, x in adaylar.items()}))
        son[k] = dict(adaylar[kazanan_ad]["kayit"], adaylar=adaylar)
    return son

def maci_bul(maclar, takim):
    """Takım adına göre maçı bulur. 'Danimarka' hem 'Danimarka' hem 'Danimarka/Galler' (kombine)
    ile eşleşebilir: önce tam ad eşleşmesi, sonra '/' içermeyen, sonra en çok marketli olan seçilir."""
    t = takim.casefold().strip()
    aday = [m for m in maclar if t in str(m.get("HN", "")).casefold() or t in str(m.get("AN", "")).casefold()]
    if len(aday) <= 1:
        return aday
    tam = [m for m in aday if t in (str(m.get("HN", "")).casefold().strip(), str(m.get("AN", "")).casefold().strip())]
    if tam:
        aday = tam
    tekil = [m for m in aday if "/" not in str(m.get("HN", "")) + str(m.get("AN", ""))]
    if tekil:
        aday = tekil
    return [max(aday, key=lambda m: len(m.get("MA") or []))]

def isim_ekle(df, sozluk):
    """Uzun tabloya market_adi ve secenek_adi sütunlarını ekler."""
    def bul(r):
        sov = r["ozel_deger"]
        sov_ok = sov is not None and sov == sov and sov != 0
        k1, k2 = f"{r['market_id']}|{sov}", f"{r['market_id']}|*"
        g = sozluk.get(k1) or sozluk.get(k2)
        varsayilan_sec = r.get("json_etiket") or f"seçenek {r['secenek_no']}"
        # isimler öğrenildiği spora aittir (hazır sözlük futbol = 1); başka sporda aynı kod başka market olabilir
        spor = r.get("spor_kodu")
        if g and spor is not None and spor == spor and int(spor) != int(g.get("spor", 1)):
            g = None
        if not g:
            return pd.Series([f"market {r['market_id']}" + (f" ({sov:g})" if sov_ok else ""), varsayilan_sec])
        ad = g["ad"]
        hnd = re.search(r"\((\d+(?:,\d+)?):(\d+(?:,\d+)?)\)", ad)
        if hnd and sov_ok:
            # handikap: pozitif çizgi ev sahibine (x:0), negatif deplasmana (0:x)
            x = f"{abs(sov):g}".replace(".", ",")
            yeni_h = f"({x}:0)" if sov > 0 else f"(0:{x})"
            ad = ad[:hnd.start()] + yeni_h + ad[hnd.end():]
        elif "{cizgi}" in ad:
            ad = ad.replace("{cizgi}", f"{sov:g}".replace(".", ",") if sov_ok else "?")
        elif k1 not in sozluk and sov_ok and not hnd:
            ad = f"{ad} [{sov:g}]"
        if r.get("json_etiket"):
            sec = r["json_etiket"]
        elif g.get("maca_ozel") and g.get("ogrenilen_mac") != r["kod"]:
            sec = varsayilan_sec
        else:
            sec = g["secenekler"].get(str(r["secenek_no"]), varsayilan_sec)
            sec = sec.replace("{ev}", str(r["ev"])).replace("{dep}", str(r["deplasman"]))
        return pd.Series([ad, sec])
    d = df.copy()
    d[["market_adi", "secenek_adi"]] = d.apply(bul, axis=1)
    return d

def sozluk_ogren_ortak(ornekler, esik=0.08):
    """Birden çok örnek maçı BİRLİKTE kullanarak öğrenir.
    ornekler: [(mac_json, referans_listesi), ...]
    Her market kodu (MTID|çizgi) için, bütün örneklerde birlikte en iyi tutan isim seçilir (bire bir atama).
    Seçenekleri maçtan maça değişen marketler (oyuncu, kombo vb.) her maç için ayrı, oran değerine göre etiketlenir."""
    import numpy as np
    from collections import defaultdict
    maliyet = defaultdict(list)        # (kod, isim) -> [fark, ...]
    mac_kodlari, mac_isimleri, aday_ciftler = [], [], set()
    for mac, referans in ornekler:
        js = {}
        for mk in mac.get("MA") or []:
            k = f"{mk.get('MTID')}|{float(mk.get('SOV') or 0)}"
            js[k] = (mk, sorted([(o.get("N"), _f(o.get("O"))) for o in mk.get("OCA") or []], key=lambda x: x[0]))
        refs = {rm["ad"]: rm for rm in referans if not rm["ad"].startswith(OZEL_KELIMELER)}
        mac_kodlari.append(js); mac_isimleri.append(refs)
        for n, rm in refs.items():
            for k, (mk, sec) in js.items():
                if _cizgi_uyumlu(n, mk.get("SOV")) and len(sec) == len(rm["secenekler"]):
                    aday_ciftler.add((k, n))
    # Her aday çift için her örnek maçta kanıt topla:
    #  ikisi de varsa sıralı oran farkı; biri var biri yoksa ceza (metinler eksiksiz kabul edilir)
    YOKLUK_CEZASI = 0.10
    for k, n in aday_ciftler:
        for js, refs in zip(mac_kodlari, mac_isimleri):
            kv, nv = k in js, n in refs
            if kv and nv:
                r = _sirali_fark(refs[n]["secenekler"], js[k][1])
                maliyet[(k, n)].append(r[0] if r else 1.0)
            elif kv != nv:
                maliyet[(k, n)].append(YOKLUK_CEZASI)
    sayim_isim = {n for refs in mac_isimleri for n in refs}
    kodlar = sorted({k for k, _ in maliyet}); isimler = sorted({n for _, n in maliyet})
    BUYUK = 9.0
    M = np.full((len(kodlar), len(isimler)), BUYUK)
    destek = np.zeros_like(M)
    for (k, n), lst in maliyet.items():
        i, j = kodlar.index(k), isimler.index(n)
        M[i, j] = float(np.mean(lst)); destek[i, j] = sum(1 for x in lst if x != YOKLUK_CEZASI)
    from scipy.optimize import linear_sum_assignment
    satir, sutun = linear_sum_assignment(M)
    sozluk, rapor = {}, []
    for i, j in zip(satir, sutun):
        if M[i, j] > esik:
            continue
        k, n = kodlar[i], isimler[j]
        # belirsizlik: bu isim için ikinci en iyi kod / bu kod için ikinci en iyi isim ne kadar yakın?
        sira_k = np.sort(M[:, j]); sira_n = np.sort(M[i, :])
        pay = min(sira_k[1] - sira_k[0] if len(sira_k) > 1 else 1, sira_n[1] - sira_n[0] if len(sira_n) > 1 else 1)
        mtid, sov = k.split("|")
        # seçenek etiketleri: sıralı (ilk örnekteki seçenek listesi)
        ref_sec = next(rm["secenekler"] for _, referans in ornekler for rm in referans if rm["ad"] == n)
        nler = None
        for mac, _ in ornekler:
            for mk in mac.get("MA") or []:
                if f"{mk.get('MTID')}|{float(mk.get('SOV') or 0)}" == k:
                    nler = sorted(o.get("N") for o in mk.get("OCA") or [])
        sec_map = {str(nn): e for nn, (e, _) in zip(nler, ref_sec)}
        spor_ornek = next((m.get("TYPE") for m, _ in ornekler if m.get("TYPE") is not None), 1)
        kayit = {"spor": spor_ornek, "ad": n, "secenekler": sec_map, "fark": round(float(M[i, j]), 4),
                 "supheli": bool(pay < 0.02), "maca_ozel": False, "destek": int(destek[i, j]), "yontem": "ortak"}
        sozluk[k] = kayit
        sozluk.setdefault(f"{mtid}|*", dict(kayit, ad=_sablon(n, float(sov))))
        rapor.append({"kod": k, "market": n, "ornek_sayisi": int(destek[i, j]),
                      "ortanca_fark_%": round(float(M[i, j]) * 100, 1), "fark_payi_%": round(float(pay) * 100, 1)})
    # 2. tur: sırası sitedekinden farklı olan sabit marketler (ör. 'Maç Sonucu ve Karşılıklı Gol') -> değere göre, yine ortak
    atanan_kod = {r["kod"] for r in rapor}; atanan_isim = {r["market"] for r in rapor}
    kalan_isim = sorted({n for refs in mac_isimleri for n in refs} - atanan_isim)
    maliyet2, etiket2 = defaultdict(list), {}
    for js, refs in zip(mac_kodlari, mac_isimleri):
        for n in kalan_isim:
            for k, (mk, sec) in js.items():
                if k in atanan_kod or not _cizgi_uyumlu(n, mk.get("SOV")):
                    continue
                if n in refs and len(sec) == len(refs[n]["secenekler"]):
                    r = _eslestir(refs[n]["secenekler"], sec)
                    if r:
                        maliyet2[(k, n)].append(r[0])
                        if (k, n) not in etiket2 or r[0] < etiket2[(k, n)][0]:
                            etiket2[(k, n)] = (r[0], r[1])
    for (k, n) in list(maliyet2):
        for js, refs in zip(mac_kodlari, mac_isimleri):
            if (k in js) != (n in refs):
                maliyet2[(k, n)].append(YOKLUK_CEZASI)
    if maliyet2:
        k2 = sorted({k for k, _ in maliyet2}); n2 = sorted({n for _, n in maliyet2})
        M2 = np.full((len(k2), len(n2)), BUYUK)
        for (k, n), lst in maliyet2.items():
            M2[k2.index(k), n2.index(n)] = float(np.mean(lst))
        for i, j in zip(*linear_sum_assignment(M2)):
            if M2[i, j] > esik:
                continue
            k, n = k2[i], n2[j]
            mtid, sov = k.split("|")
            # seçenek eşleşmesi: örneklerin önerdiği eşlemelerden, BÜTÜN örneklerde toplamda en iyi tutanı seç
            adaylar_map = []
            for js, refs in zip(mac_kodlari, mac_isimleri):
                if k in js and n in refs:
                    r = _eslestir(refs[n]["secenekler"], js[k][1])
                    if r: adaylar_map.append(r[1])
            def toplam(m):
                t = 0.0
                for js, refs in zip(mac_kodlari, mac_isimleri):
                    if k in js and n in refs:
                        ro = dict(refs[n]["secenekler"])
                        for nn, o in js[k][1]:
                            e = m.get(nn)
                            if o and e and ro.get(e):
                                t += abs(ro[e] - o) / ro[e]
                return t
            en_iyi_map = min(adaylar_map, key=toplam)
            sec_map = {str(nn): e for nn, e in en_iyi_map.items()}
            kayit = {"spor": next((m.get("TYPE") for m, _ in ornekler if m.get("TYPE") is not None), 1),
                     "ad": n, "secenekler": sec_map, "fark": round(float(M2[i, j]), 4), "supheli": False,
                     "maca_ozel": False, "destek": len([x for x in maliyet2[(k, n)] if x != YOKLUK_CEZASI]), "yontem": "ortak-değer"}
            sozluk[k] = kayit
            sozluk.setdefault(f"{mtid}|*", dict(kayit, ad=_sablon(n, float(sov))))
            rapor.append({"kod": k, "market": n, "ornek_sayisi": kayit["destek"],
                          "ortanca_fark_%": round(float(M2[i, j]) * 100, 1), "fark_payi_%": None})
    # maça özel marketler: her örnekte ayrı, değere göre
    for mac, referans in ornekler:
        ozel = [rm for rm in referans if rm["ad"].startswith(OZEL_KELIMELER)]
        if not ozel:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            s2, _, _ = sozluk_ogren(mac, ozel)
        for k, v in s2.items():
            if not k.endswith("|*"):
                v = dict(v, maca_ozel=True, spor=mac.get("TYPE") or 1)
                sozluk.setdefault(k, v)
                sozluk.setdefault(f"{k.split('|')[0]}|*", v)
    atanmayan = sorted(set(sayim_isim) - {r["market"] for r in rapor})
    return sozluk, pd.DataFrame(rapor), atanmayan


In [ ]:
# 2) Bülteni çek
BULTEN_URL = URL   # Plan B başka bir adres bulursa buraya yapıştır

r = requests.get(BULTEN_URL, headers=HEADERS, timeout=30)
print("HTTP:", r.status_code, "| boyut:", len(r.content) // 1024, "KB")
veri = r.json()
tum_maclar = mac_listesini_bul(veri)
print("Bültendeki toplam maç:", len(tum_maclar))
if tum_maclar:
    print("Örnek maç alanları:", {k: v for k, v in tum_maclar[0].items() if k != "MA"})

In [ ]:
# 3) Market isimleri
# Yeni örnek eklemek istersen: ("takım adı", r'''sitedeki maç sayfasından kopyalanan metin'''),
# Metni kopyaladıktan hemen sonra çalıştır; maç başlamışsa bültende olmaz ve atlanır.
REFERANSLAR = [
]

# 17.11.2026 gerçek bülten verisi + 4 örnek maçla (Ukrayna-Gürcistan, Poole Town, Danimarka-Norveç, Liberya) doğrulanmış sözlük
HAZIR_SOZLUK = json.loads(r'''{"11|1.5": {"ad": "1,5 Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.052, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "11|*": {"ad": "{cizgi} Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.052, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "12|2.5": {"ad": "2,5 Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.028, "supheli": false, "maca_ozel": false, "destek": 4, "yontem": "ortak"}, "12|*": {"ad": "{cizgi} Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.028, "supheli": false, "maca_ozel": false, "destek": 4, "yontem": "ortak"}, "13|3.5": {"ad": "3,5 Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0652, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "13|*": {"ad": "{cizgi} Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0652, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "14|1.5": {"ad": "1. Yarı 1,5 Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0302, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "14|*": {"ad": "1. Yarı {cizgi} Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0302, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "155|4.5": {"ad": "4,5 Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0675, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "155|*": {"ad": "{cizgi} Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0675, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "15|2.5": {"ad": "1. Yarı 2,5 Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0498, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "15|*": {"ad": "1. Yarı {cizgi} Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0498, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "1|0.0": {"ad": "Maç Sonucu", "secenekler": {"1": "MS 1", "2": "MS X", "3": "MS 2"}, "fark": 0.0013, "supheli": false, "maca_ozel": false, "destek": 4, "yontem": "ortak"}, "1|*": {"ad": "Maç Sonucu", "secenekler": {"1": "MS 1", "2": "MS X", "3": "MS 2"}, "fark": 0.0013, "supheli": false, "maca_ozel": false, "destek": 4, "yontem": "ortak"}, "207|0.5": {"ad": "0,5 Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.041, "supheli": true, "maca_ozel": false, "destek": 1, "yontem": "ortak"}, "207|*": {"ad": "{cizgi} Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.041, "supheli": true, "maca_ozel": false, "destek": 1, "yontem": "ortak"}, "209|0.5": {"ad": "1. Yarı 0,5 Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0259, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "209|*": {"ad": "1. Yarı {cizgi} Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0259, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "20|1.5": {"ad": "Ev Sahibi 1,5 Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0221, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "20|*": {"ad": "Ev Sahibi {cizgi} Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0221, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "212|0.5": {"ad": "Ev Sahibi 0,5 Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0362, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "212|*": {"ad": "Ev Sahibi {cizgi} Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0362, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "216|10.5": {"ad": "10,5 Korner Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "216|*": {"ad": "{cizgi} Korner Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "216|11.5": {"ad": "11,5 Korner Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 1, "yontem": "ortak"}, "216|6.5": {"ad": "6,5 Korner Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 1, "yontem": "ortak"}, "216|7.5": {"ad": "7,5 Korner Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "216|8.5": {"ad": "8,5 Korner Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "216|9.5": {"ad": "9,5 Korner Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "218|3.5": {"ad": "1. Yarı 3,5 Korner Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "218|*": {"ad": "1. Yarı {cizgi} Korner Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "218|4.5": {"ad": "1. Yarı 4,5 Korner Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "218|5.5": {"ad": "1. Yarı 5,5 Korner Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "220|0.0": {"ad": "En Çok Korner", "secenekler": {"1": "Ev.", "2": "X", "3": "Dep."}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "220|*": {"ad": "En Çok Korner", "secenekler": {"1": "Ev.", "2": "X", "3": "Dep."}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "222|0.0": {"ad": "1. Yarı En Çok Korner", "secenekler": {"1": "Ev.", "2": "X", "3": "Dep."}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "222|*": {"ad": "1. Yarı En Çok Korner", "secenekler": {"1": "Ev.", "2": "X", "3": "Dep."}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "224|1.0": {"ad": "İlk Korner", "secenekler": {"1": "1", "2": "Olmaz", "3": "2"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "224|*": {"ad": "İlk Korner", "secenekler": {"1": "1", "2": "Olmaz", "3": "2"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "225|0.0": {"ad": "Kırmızı Kart", "secenekler": {"1": "Var", "2": "Yok"}, "fark": 0.027, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "225|*": {"ad": "Kırmızı Kart", "secenekler": {"1": "Var", "2": "Yok"}, "fark": 0.027, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "256|0.5": {"ad": "Deplasman 0,5 Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0458, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "256|*": {"ad": "Deplasman {cizgi} Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0458, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "268|-1.0": {"ad": "Handikaplı Maç Sonucu (0:1)", "secenekler": {"1": "HMS 1", "2": "HMS X", "3": "HMS 2"}, "fark": 0.0071, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "268|*": {"ad": "Handikaplı Maç Sonucu (0:1)", "secenekler": {"1": "HMS 1", "2": "HMS X", "3": "HMS 2"}, "fark": 0.0071, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "268|-2.0": {"ad": "Handikaplı Maç Sonucu (0:2)", "secenekler": {"1": "HMS 1", "2": "HMS X", "3": "HMS 2"}, "fark": 0.0586, "supheli": false, "maca_ozel": false, "destek": 1, "yontem": "ortak"}, "268|1.0": {"ad": "Handikaplı Maç Sonucu (1:0)", "secenekler": {"1": "HMS 1", "2": "HMS X", "3": "HMS 2"}, "fark": 0.0122, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "268|2.0": {"ad": "Handikaplı Maç Sonucu (2:0)", "secenekler": {"1": "HMS 1", "2": "HMS X", "3": "HMS 2"}, "fark": 0.0211, "supheli": false, "maca_ozel": false, "destek": 1, "yontem": "ortak"}, "272|3.5": {"ad": "Maç Sonucu ve 3,5 Alt/Üst", "secenekler": {"1": "MS1 & Alt", "2": "MSX & Alt", "3": "MS2 & Alt", "4": "MS1 & Üst", "5": "MSX & Üst", "6": "MS2 & Üst"}, "fark": 0.0469, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "272|*": {"ad": "Maç Sonucu ve {cizgi} Alt/Üst", "secenekler": {"1": "MS1 & Alt", "2": "MSX & Alt", "3": "MS2 & Alt", "4": "MS1 & Üst", "5": "MSX & Üst", "6": "MS2 & Üst"}, "fark": 0.0469, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "272|4.5": {"ad": "Maç Sonucu ve 4,5 Alt/Üst", "secenekler": {"1": "MS1 & Alt", "2": "MSX & Alt", "3": "MS2 & Alt", "4": "MS1 & Üst", "5": "MSX & Üst", "6": "MS2 & Üst"}, "fark": 0.0514, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "291|0.0": {"ad": "İlk Golü Hangi Takım Atar?", "secenekler": {"1": "1", "2": "Olmaz", "3": "2"}, "fark": 0.03, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "291|*": {"ad": "İlk Golü Hangi Takım Atar?", "secenekler": {"1": "1", "2": "Olmaz", "3": "2"}, "fark": 0.03, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "295|0.0": {"ad": "Ev Sahibi İki Yarıda da Gol", "secenekler": {"1": "Atar", "2": "Atamaz"}, "fark": 0.0344, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "295|*": {"ad": "Ev Sahibi İki Yarıda da Gol", "secenekler": {"1": "Atar", "2": "Atamaz"}, "fark": 0.0344, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "296|0.0": {"ad": "Deplasman İki Yarıda da Gol", "secenekler": {"1": "Atar", "2": "Atamaz"}, "fark": 0.0329, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "296|*": {"ad": "Deplasman İki Yarıda da Gol", "secenekler": {"1": "Atar", "2": "Atamaz"}, "fark": 0.0329, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "299|0.0": {"ad": "Korner Tek/Çift", "secenekler": {"1": "Tek", "2": "Çift"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "299|*": {"ad": "Korner Tek/Çift", "secenekler": {"1": "Tek", "2": "Çift"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "29|1.5": {"ad": "Deplasman 1,5 Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0218, "supheli": false, "maca_ozel": false, "destek": 4, "yontem": "ortak"}, "29|*": {"ad": "Deplasman {cizgi} Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0218, "supheli": false, "maca_ozel": false, "destek": 4, "yontem": "ortak"}, "301|3.5": {"ad": "3,5 Kart Puanı Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0425, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "301|*": {"ad": "{cizgi} Kart Puanı Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0425, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "301|4.5": {"ad": "4,5 Kart Puanı Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0442, "supheli": true, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "301|5.5": {"ad": "5,5 Kart Puanı Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "326|2.5": {"ad": "Ev Sahibi 2,5 Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0468, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "326|*": {"ad": "Ev Sahibi {cizgi} Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0468, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "328|2.5": {"ad": "Deplasman 2,5 Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0501, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "328|*": {"ad": "Deplasman {cizgi} Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0501, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "338|0.0": {"ad": "Toplam Korner Aralığı", "secenekler": {"1": "0-8", "2": "9-11", "3": "12+"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "338|*": {"ad": "Toplam Korner Aralığı", "secenekler": {"1": "0-8", "2": "9-11", "3": "12+"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "340|0.0": {"ad": "1. Yarı Korner Aralığı", "secenekler": {"1": "0-4", "2": "5-6", "3": "7+"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "340|*": {"ad": "1. Yarı Korner Aralığı", "secenekler": {"1": "0-4", "2": "5-6", "3": "7+"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "342|1.5": {"ad": "Maç Sonucu ve 1,5 Alt/Üst", "secenekler": {"1": "MS1 & Alt", "2": "MSX & Alt", "3": "MS2 & Alt", "4": "MS1 & Üst", "5": "MSX & Üst", "6": "MS2 & Üst"}, "fark": 0.0647, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "342|*": {"ad": "Maç Sonucu ve {cizgi} Alt/Üst", "secenekler": {"1": "MS1 & Alt", "2": "MSX & Alt", "3": "MS2 & Alt", "4": "MS1 & Üst", "5": "MSX & Üst", "6": "MS2 & Üst"}, "fark": 0.0647, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "38|0.0": {"ad": "Karşılıklı Gol", "secenekler": {"1": "Var", "2": "Yok"}, "fark": 0.0258, "supheli": false, "maca_ozel": false, "destek": 4, "yontem": "ortak"}, "38|*": {"ad": "Karşılıklı Gol", "secenekler": {"1": "Var", "2": "Yok"}, "fark": 0.0258, "supheli": false, "maca_ozel": false, "destek": 4, "yontem": "ortak"}, "3|0.0": {"ad": "Çifte Şans", "secenekler": {"1": "ÇŞ 1-X", "2": "ÇŞ 1-2", "3": "ÇŞ X-2"}, "fark": 0.0011, "supheli": false, "maca_ozel": false, "destek": 4, "yontem": "ortak"}, "3|*": {"ad": "Çifte Şans", "secenekler": {"1": "ÇŞ 1-X", "2": "ÇŞ 1-2", "3": "ÇŞ X-2"}, "fark": 0.0011, "supheli": false, "maca_ozel": false, "destek": 4, "yontem": "ortak"}, "43|0.0": {"ad": "Toplam Gol Aralığı", "secenekler": {"1": "0-1 Gol", "2": "2-3 Gol", "3": "4-5 Gol", "4": "6+ Gol"}, "fark": 0.0607, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "43|*": {"ad": "Toplam Gol Aralığı", "secenekler": {"1": "0-1 Gol", "2": "2-3 Gol", "3": "4-5 Gol", "4": "6+ Gol"}, "fark": 0.0607, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "446|2.5": {"ad": "2,5 Alt/Üst ve Karşılıklı Gol", "secenekler": {"1": "Alt & Var", "2": "Üst & Var", "3": "Alt & Yok", "4": "Üst & Yok"}, "fark": 0.031, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "446|*": {"ad": "{cizgi} Alt/Üst ve Karşılıklı Gol", "secenekler": {"1": "Alt & Var", "2": "Üst & Var", "3": "Alt & Yok", "4": "Üst & Yok"}, "fark": 0.031, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "450|0.0": {"ad": "1. Yarı Tek/Çift", "secenekler": {"1": "Tek", "2": "Çift"}, "fark": 0.0068, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "450|*": {"ad": "1. Yarı Tek/Çift", "secenekler": {"1": "Tek", "2": "Çift"}, "fark": 0.0068, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "452|0.0": {"ad": "1. Yarı Karşılıklı Gol", "secenekler": {"1": "Var", "2": "Yok"}, "fark": 0.0362, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "452|*": {"ad": "1. Yarı Karşılıklı Gol", "secenekler": {"1": "Var", "2": "Yok"}, "fark": 0.0362, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "455|0.5": {"ad": "Ev Sahibi 1.Y 0,5 Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0218, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "455|*": {"ad": "Ev Sahibi 1.Y {cizgi} Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0218, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "457|0.5": {"ad": "Deplasman 1.Y 0,5 Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0242, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "457|*": {"ad": "Deplasman 1.Y {cizgi} Gol Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0242, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "459|1.5": {"ad": "1.Y Sonucu ve 1.Y 1,5 Alt/Üst", "secenekler": {"1": "1.Y 1 & Alt", "2": "1.Y X & Alt", "3": "1.Y 2 & Alt", "4": "1.Y 1 & Üst", "5": "1.Y X & Üst", "6": "1.Y 2 & Üst"}, "fark": 0.0304, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "459|*": {"ad": "1.Y Sonucu ve 1.Y {cizgi} Alt/Üst", "secenekler": {"1": "1.Y 1 & Alt", "2": "1.Y X & Alt", "3": "1.Y 2 & Alt", "4": "1.Y 1 & Üst", "5": "1.Y X & Üst", "6": "1.Y 2 & Üst"}, "fark": 0.0304, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "48|0.0": {"ad": "En Çok Gol Olacak Yarı", "secenekler": {"1": "1. Y", "2": "Eşit", "3": "2. Y"}, "fark": 0.0056, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "48|*": {"ad": "En Çok Gol Olacak Yarı", "secenekler": {"1": "1. Y", "2": "Eşit", "3": "2. Y"}, "fark": 0.0056, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "49|0.0": {"ad": "Tek/Çift", "secenekler": {"1": "Tek", "2": "Çift"}, "fark": 0.0038, "supheli": false, "maca_ozel": false, "destek": 4, "yontem": "ortak"}, "49|*": {"ad": "Tek/Çift", "secenekler": {"1": "Tek", "2": "Çift"}, "fark": 0.0038, "supheli": false, "maca_ozel": false, "destek": 4, "yontem": "ortak"}, "528|0.0": {"ad": "İki Yarı da 1,5 Alt", "secenekler": {"1": "Evet", "2": "Hayır"}, "fark": 0.0358, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "528|*": {"ad": "İki Yarı da 1,5 Alt", "secenekler": {"1": "Evet", "2": "Hayır"}, "fark": 0.0358, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "529|0.0": {"ad": "İki Yarı da 1,5 Üst", "secenekler": {"1": "Evet", "2": "Hayır"}, "fark": 0.0562, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "529|*": {"ad": "İki Yarı da 1,5 Üst", "secenekler": {"1": "Evet", "2": "Hayır"}, "fark": 0.0562, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "571|0.0": {"ad": "İlk Yarı / Maç Skoru", "secenekler": {"1": "0:0 / 0:0", "2": "0:0 / 0:1", "3": "0:0 / 0:2", "4": "0:0 / 0:3", "5": "0:0 / 1:0", "6": "0:0 / 1:1", "7": "0:0 / 1:2", "8": "0:0 / 2:0", "9": "0:0 / 2:1", "10": "0:0 / 3:0", "11": "0:0 / 4+", "12": "1:0 / 1:0", "13": "1:0 / 1:1", "14": "1:0 / 1:2", "15": "1:0 / 2:0", "16": "1:0 / 2:1", "17": "1:0 / 3:0", "18": "1:0 / 4+", "19": "0:1 / 0:1", "20": "0:1 / 0:2", "21": "0:1 / 0:3", "22": "0:1 / 1:1", "23": "0:1 / 1:2", "24": "0:1 / 2:1", "25": "0:1 / 4+", "26": "2:0 / 2:0", "27": "2:0 / 2:1", "28": "2:0 / 3:0", "29": "2:0 / 4+", "30": "1:1 / 1:1", "31": "1:1 / 1:2", "32": "1:1 / 2:1", "33": "1:1 / 4+", "34": "0:2 / 0:2", "35": "0:2 / 0:3", "36": "0:2 / 1:2", "37": "0:2 / 4+", "38": "3:0 / 3:0", "39": "3:0 / 4+", "40": "2:1 / 2:1", "41": "2:1 / 4+", "42": "1:2 / 1:2", "43": "1:2 / 4+", "44": "0:3 / 0:3", "45": "0:3 / 4+", "46": "4+ / 4+"}, "fark": 0.0365, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "571|*": {"ad": "İlk Yarı / Maç Skoru", "secenekler": {"1": "0:0 / 0:0", "2": "0:0 / 0:1", "3": "0:0 / 0:2", "4": "0:0 / 0:3", "5": "0:0 / 1:0", "6": "0:0 / 1:1", "7": "0:0 / 1:2", "8": "0:0 / 2:0", "9": "0:0 / 2:1", "10": "0:0 / 3:0", "11": "0:0 / 4+", "12": "1:0 / 1:0", "13": "1:0 / 1:1", "14": "1:0 / 1:2", "15": "1:0 / 2:0", "16": "1:0 / 2:1", "17": "1:0 / 3:0", "18": "1:0 / 4+", "19": "0:1 / 0:1", "20": "0:1 / 0:2", "21": "0:1 / 0:3", "22": "0:1 / 1:1", "23": "0:1 / 1:2", "24": "0:1 / 2:1", "25": "0:1 / 4+", "26": "2:0 / 2:0", "27": "2:0 / 2:1", "28": "2:0 / 3:0", "29": "2:0 / 4+", "30": "1:1 / 1:1", "31": "1:1 / 1:2", "32": "1:1 / 2:1", "33": "1:1 / 4+", "34": "0:2 / 0:2", "35": "0:2 / 0:3", "36": "0:2 / 1:2", "37": "0:2 / 4+", "38": "3:0 / 3:0", "39": "3:0 / 4+", "40": "2:1 / 2:1", "41": "2:1 / 4+", "42": "1:2 / 1:2", "43": "1:2 / 4+", "44": "0:3 / 0:3", "45": "0:3 / 4+", "46": "4+ / 4+"}, "fark": 0.0365, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "584|0.0": {"ad": "Ev Sahibi Herhangi Bir Yarıyı Kazanır", "secenekler": {"1": "Evet", "2": "Hayır"}, "fark": 0.0086, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "584|*": {"ad": "Ev Sahibi Herhangi Bir Yarıyı Kazanır", "secenekler": {"1": "Evet", "2": "Hayır"}, "fark": 0.0086, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "585|0.0": {"ad": "Deplasman Herhangi Bir Yarıyı Kazanır", "secenekler": {"1": "Evet", "2": "Hayır"}, "fark": 0.0072, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "585|*": {"ad": "Deplasman Herhangi Bir Yarıyı Kazanır", "secenekler": {"1": "Evet", "2": "Hayır"}, "fark": 0.0072, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "586|0.0": {"ad": "Ev Sahibi Hangi Yarıda Daha Çok Gol Atar?", "secenekler": {"1": "1.", "2": "Eşit", "3": "2."}, "fark": 0.0142, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "586|*": {"ad": "Ev Sahibi Hangi Yarıda Daha Çok Gol Atar?", "secenekler": {"1": "1.", "2": "Eşit", "3": "2."}, "fark": 0.0142, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "587|0.0": {"ad": "Deplasman Hangi Yarıda Daha Çok Gol Atar?", "secenekler": {"1": "1.", "2": "Eşit", "3": "2."}, "fark": 0.0145, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "587|*": {"ad": "Deplasman Hangi Yarıda Daha Çok Gol Atar?", "secenekler": {"1": "1.", "2": "Eşit", "3": "2."}, "fark": 0.0145, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "588|0.0": {"ad": "Hangi Takım Kaç Farkla Kazanır?", "secenekler": {"1": "Ev. 3+", "2": "Ev. 2", "3": "Ev. 1", "4": "Dep. 1", "5": "Dep. 2", "6": "Dep. 3+", "7": "0"}, "fark": 0.0181, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "588|*": {"ad": "Hangi Takım Kaç Farkla Kazanır?", "secenekler": {"1": "Ev. 3+", "2": "Ev. 2", "3": "Ev. 1", "4": "Dep. 1", "5": "Dep. 2", "6": "Dep. 3+", "7": "0"}, "fark": 0.0181, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "591|0.0": {"ad": "Ev Sahibi Her İki Yarıyı da Kazanır", "secenekler": {"1": "Evet", "2": "Hayır"}, "fark": 0.019, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "591|*": {"ad": "Ev Sahibi Her İki Yarıyı da Kazanır", "secenekler": {"1": "Evet", "2": "Hayır"}, "fark": 0.019, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "592|0.0": {"ad": "Deplasman Her İki Yarıyı da Kazanır", "secenekler": {"1": "Evet", "2": "Hayır"}, "fark": 0.0194, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "592|*": {"ad": "Deplasman Her İki Yarıyı da Kazanır", "secenekler": {"1": "Evet", "2": "Hayır"}, "fark": 0.0194, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "593|0.0": {"ad": "Penaltı Olur Mu?", "secenekler": {"1": "Evet", "2": "Hayır"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "593|*": {"ad": "Penaltı Olur Mu?", "secenekler": {"1": "Evet", "2": "Hayır"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "599|0.0": {"ad": "2. Yarı Karşılıklı Gol", "secenekler": {"1": "Var", "2": "Yok"}, "fark": 0.0311, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "599|*": {"ad": "2. Yarı Karşılıklı Gol", "secenekler": {"1": "Var", "2": "Yok"}, "fark": 0.0311, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "5|0.0": {"ad": "İlk Yarı/Maç Sonucu", "secenekler": {"1": "1/1", "2": "1/X", "3": "1/2", "4": "X/1", "5": "X/X", "6": "X/2", "7": "2/1", "8": "2/X", "9": "2/2"}, "fark": 0.0194, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "5|*": {"ad": "İlk Yarı/Maç Sonucu", "secenekler": {"1": "1/1", "2": "1/X", "3": "1/2", "4": "X/1", "5": "X/X", "6": "X/2", "7": "2/1", "8": "2/X", "9": "2/2"}, "fark": 0.0194, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "601|4.5": {"ad": "Ev Sahibi 4,5 Korner Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "601|*": {"ad": "Ev Sahibi {cizgi} Korner Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "602|3.5": {"ad": "Deplasman 3,5 Korner Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 1, "yontem": "ortak"}, "602|*": {"ad": "Deplasman {cizgi} Korner Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 1, "yontem": "ortak"}, "602|4.5": {"ad": "Deplasman 4,5 Korner Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 1, "yontem": "ortak"}, "603|0.0": {"ad": "Hangi Takım Daha Çok Kart Puanı Alır?", "secenekler": {"1": "Ev.", "2": "Eşit", "3": "Dep."}, "fark": 0.0081, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "603|*": {"ad": "Hangi Takım Daha Çok Kart Puanı Alır?", "secenekler": {"1": "Ev.", "2": "Eşit", "3": "Dep."}, "fark": 0.0081, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "777|0.0": {"ad": "Maç Skoru", "secenekler": {"1": "1:0", "2": "2:0", "3": "2:1", "4": "3:0", "5": "3:1", "6": "3:2", "7": "4:0", "8": "4:1", "9": "4:2", "10": "5:0", "11": "5:1", "12": "6:0", "13": "0:0", "14": "1:1", "15": "2:2", "16": "3:3", "17": "0:1", "18": "0:2", "19": "1:2", "20": "0:3", "21": "1:3", "22": "2:3", "23": "0:4", "24": "1:4", "25": "2:4", "26": "0:5", "27": "1:5", "28": "0:6", "29": "diğer"}, "fark": 0.0445, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "777|*": {"ad": "Maç Skoru", "secenekler": {"1": "1:0", "2": "2:0", "3": "2:1", "4": "3:0", "5": "3:1", "6": "3:2", "7": "4:0", "8": "4:1", "9": "4:2", "10": "5:0", "11": "5:1", "12": "6:0", "13": "0:0", "14": "1:1", "15": "2:2", "16": "3:3", "17": "0:1", "18": "0:2", "19": "1:2", "20": "0:3", "21": "1:3", "22": "2:3", "23": "0:4", "24": "1:4", "25": "2:4", "26": "0:5", "27": "1:5", "28": "0:6", "29": "diğer"}, "fark": 0.0445, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "798|-0.5": {"ad": "Korner Handikap (0:0,5)", "secenekler": {"1": "1", "2": "2"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "798|*": {"ad": "Korner Handikap (0:0,5)", "secenekler": {"1": "1", "2": "2"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "798|-1.5": {"ad": "Korner Handikap (0:1,5)", "secenekler": {"1": "1", "2": "2"}, "fark": 0.05, "supheli": true, "maca_ozel": false, "destek": 1, "yontem": "ortak"}, "798|0.5": {"ad": "Korner Handikap (0,5:0)", "secenekler": {"1": "1", "2": "2"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "798|1.5": {"ad": "Korner Handikap (1,5:0)", "secenekler": {"1": "1", "2": "2"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 1, "yontem": "ortak"}, "799|-0.5": {"ad": "1. Yarı Korner Handikap (0:0,5)", "secenekler": {"1": "1", "2": "2"}, "fark": 0.05, "supheli": false, "maca_ozel": false, "destek": 1, "yontem": "ortak"}, "799|*": {"ad": "1. Yarı Korner Handikap (0:0,5)", "secenekler": {"1": "1", "2": "2"}, "fark": 0.05, "supheli": false, "maca_ozel": false, "destek": 1, "yontem": "ortak"}, "799|0.5": {"ad": "1. Yarı Korner Handikap (0,5:0)", "secenekler": {"1": "1", "2": "2"}, "fark": 0.0, "supheli": false, "maca_ozel": false, "destek": 1, "yontem": "ortak"}, "7|0.0": {"ad": "1. Yarı Sonucu", "secenekler": {"1": "1.Y 1", "2": "1.Y X", "3": "1.Y 2"}, "fark": 0.0144, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "7|*": {"ad": "1. Yarı Sonucu", "secenekler": {"1": "1.Y 1", "2": "1.Y X", "3": "1.Y 2"}, "fark": 0.0144, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "800|1.5": {"ad": "1. Yarı 1,5 Kart Puanı Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0237, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "800|*": {"ad": "1. Yarı {cizgi} Kart Puanı Alt/Üst", "secenekler": {"1": "Alt", "2": "Üst"}, "fark": 0.0237, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak"}, "8|0.0": {"ad": "1. Yarı Çifte Şans", "secenekler": {"1": "1.Y 1-X", "2": "1.Y 1-2", "3": "1.Y X-2"}, "fark": 0.0107, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "8|*": {"ad": "1. Yarı Çifte Şans", "secenekler": {"1": "1.Y 1-X", "2": "1.Y 1-2", "3": "1.Y X-2"}, "fark": 0.0107, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "9|0.0": {"ad": "2. Yarı Sonucu", "secenekler": {"1": "2.Y 1", "2": "2.Y X", "3": "2.Y 2"}, "fark": 0.0096, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "9|*": {"ad": "2. Yarı Sonucu", "secenekler": {"1": "2.Y 1", "2": "2.Y X", "3": "2.Y 2"}, "fark": 0.0096, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak"}, "343|2.5": {"ad": "Maç Sonucu ve 2,5 Alt/Üst", "secenekler": {"1": "MS1 & Alt", "2": "MSX & Alt", "3": "MS2 & Alt", "4": "MS1 & Üst", "5": "MSX & Üst", "6": "MS2 & Üst"}, "fark": 0.0486, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak-değer"}, "343|*": {"ad": "Maç Sonucu ve {cizgi} Alt/Üst", "secenekler": {"1": "MS1 & Alt", "2": "MSX & Alt", "3": "MS2 & Alt", "4": "MS1 & Üst", "5": "MSX & Üst", "6": "MS2 & Üst"}, "fark": 0.0486, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak-değer"}, "414|0.0": {"ad": "Maç Sonucu ve Karşılıklı Gol", "secenekler": {"1": "MS1 & Var", "3": "MSX & Var", "5": "MS2 & Var", "2": "MS1 & Yok", "4": "MSX & Yok", "6": "MS2 & Yok"}, "fark": 0.0382, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak-değer"}, "414|*": {"ad": "Maç Sonucu ve Karşılıklı Gol", "secenekler": {"1": "MS1 & Var", "3": "MSX & Var", "5": "MS2 & Var", "2": "MS1 & Yok", "4": "MSX & Yok", "6": "MS2 & Yok"}, "fark": 0.0382, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak-değer"}, "416|0.0": {"ad": "1. Yarı Sonucu ve 1. Yarı Karşılıklı Gol", "secenekler": {"1": "1.Y 1 & Var", "3": "1.Y X & Var", "5": "1.Y 2 & Var", "2": "1.Y 1 & Yok", "4": "1.Y X & Yok", "6": "1.Y 2 & Yok"}, "fark": 0.0336, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak-değer"}, "416|*": {"ad": "1. Yarı Sonucu ve 1. Yarı Karşılıklı Gol", "secenekler": {"1": "1.Y 1 & Var", "3": "1.Y X & Var", "5": "1.Y 2 & Var", "2": "1.Y 1 & Yok", "4": "1.Y X & Yok", "6": "1.Y 2 & Yok"}, "fark": 0.0336, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak-değer"}, "779|0.0": {"ad": "1. Yarı Skoru", "secenekler": {"4": "1:0", "5": "2:0", "6": "2:1", "1": "0:0", "2": "1:1", "3": "2:2", "7": "0:1", "8": "0:2", "9": "1:2", "10": "diğer"}, "fark": 0.0392, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak-değer"}, "779|*": {"ad": "1. Yarı Skoru", "secenekler": {"4": "1:0", "5": "2:0", "6": "2:1", "1": "0:0", "2": "1:1", "3": "2:2", "7": "0:1", "8": "0:2", "9": "1:2", "10": "diğer"}, "fark": 0.0392, "supheli": false, "maca_ozel": false, "destek": 2, "yontem": "ortak-değer"}, "801|0.0": {"ad": "1. Yarı / 2. Yarı Karşılıklı Gol", "secenekler": {"3": "Evet/Evet", "2": "Evet/Hayır", "4": "Hayır/Evet", "1": "Hayır/Hayır"}, "fark": 0.044, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak-değer"}, "801|*": {"ad": "1. Yarı / 2. Yarı Karşılıklı Gol", "secenekler": {"3": "Evet/Evet", "2": "Evet/Hayır", "4": "Hayır/Evet", "1": "Hayır/Hayır"}, "fark": 0.044, "supheli": false, "maca_ozel": false, "destek": 3, "yontem": "ortak-değer"}, "885|0.0": {"ad": "Karşılaşma Kombo Bahisleri", "secenekler": {"1": "{ev} 1+ Gol, 5+ Korner", "2": "9+ Korner, 5+ Kart", "3": "{dep} 1+ Gol, 5+ Korner", "4": "3+ Gol, 9+ Korner", "5": "3+ Gol, 9+ Korner, 3+ Kart", "6": "3+ Gol, 5+ Kart", "7": "3+ Gol, 9+ Korner, 4+ Kart", "8": "{ev} 2+ Gol, 6+ Korner", "9": "4+ Gol, 9+ Korner", "10": "4+ Gol, 9+ Korner, 4+ Kart", "11": "3+ Gol, 12+ Korner", "12": "4+ Gol, 9+ Korner, 5+ Kart", "13": "{dep} 2+ Gol, 6+ Korner", "14": "4+ Gol, 12+ Korner"}, "fark": 0.0, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303421, "yontem": "sıralı"}, "885|*": {"ad": "Karşılaşma Kombo Bahisleri", "secenekler": {"1": "{ev} 1+ Gol, 5+ Korner", "2": "9+ Korner, 5+ Kart", "3": "{dep} 1+ Gol, 5+ Korner", "4": "3+ Gol, 9+ Korner", "5": "3+ Gol, 9+ Korner, 3+ Kart", "6": "3+ Gol, 5+ Kart", "7": "3+ Gol, 9+ Korner, 4+ Kart", "8": "{ev} 2+ Gol, 6+ Korner", "9": "4+ Gol, 9+ Korner", "10": "4+ Gol, 9+ Korner, 4+ Kart", "11": "3+ Gol, 12+ Korner", "12": "4+ Gol, 9+ Korner, 5+ Kart", "13": "{dep} 2+ Gol, 6+ Korner", "14": "4+ Gol, 12+ Korner"}, "fark": 0.0, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303421, "yontem": "sıralı"}, "702|0.0": {"ad": "Oyuncu İlk Golü Atar", "secenekler": {"4": "Artem Dovbyk", "35": "Vladyslav Vanat", "34": "Roman Yaremchuk", "28": "Danylo Sikan", "12": "Igor Krasnopir", "20": "Georges Mikautadze", "39": "Budu Zivzivadze", "29": "Giorgi Guliashvili", "8": "Georgiy Sudakov", "30": "Hennadii Synchuk", "11": "Giorgi Kvernadze", "14": "Oleksiy Kashchuk", "3": "Zuriko Davitashvili", "24": "Giorgi Chakvetadze", "2": "Oleh Ocheretko", "37": "Irakli Yegoian", "23": "Yehor Nazaryna", "33": "Georgiy Tsitaishvili", "6": "Giorgi Moistsrapeshvili", "21": "Nika Gagnidze", "36": "Boris Nikolaevich Krushinsky", "13": "Varfolomieiev Ivan Oleksandrovych", "26": "Eduard Sarapii", "25": "Yehor Yarmoliuk", "16": "Nodari Lominadze", "32": "Mykhavko Taras", "22": "Vitaliy Mykolenko", "19": "Anzor Mekvabishvili", "15": "Luka Lochoshvili", "38": "I. Zabarnyi", "18": "Ilya Alekseevich Krupsky", "1": "Mykola Matvienko", "5": "Lasha Dvali", "9": "Giorgi Maisuradze", "10": "Otar Kakabadze", "17": "Saba Sazonov", "27": "Irakli Azarov", "31": "Saba Goglichidze", "7": "Jemal Tabidze"}, "fark": 0.01, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303421, "yontem": "değere göre"}, "702|*": {"ad": "Oyuncu İlk Golü Atar", "secenekler": {"4": "Artem Dovbyk", "35": "Vladyslav Vanat", "34": "Roman Yaremchuk", "28": "Danylo Sikan", "12": "Igor Krasnopir", "20": "Georges Mikautadze", "39": "Budu Zivzivadze", "29": "Giorgi Guliashvili", "8": "Georgiy Sudakov", "30": "Hennadii Synchuk", "11": "Giorgi Kvernadze", "14": "Oleksiy Kashchuk", "3": "Zuriko Davitashvili", "24": "Giorgi Chakvetadze", "2": "Oleh Ocheretko", "37": "Irakli Yegoian", "23": "Yehor Nazaryna", "33": "Georgiy Tsitaishvili", "6": "Giorgi Moistsrapeshvili", "21": "Nika Gagnidze", "36": "Boris Nikolaevich Krushinsky", "13": "Varfolomieiev Ivan Oleksandrovych", "26": "Eduard Sarapii", "25": "Yehor Yarmoliuk", "16": "Nodari Lominadze", "32": "Mykhavko Taras", "22": "Vitaliy Mykolenko", "19": "Anzor Mekvabishvili", "15": "Luka Lochoshvili", "38": "I. Zabarnyi", "18": "Ilya Alekseevich Krupsky", "1": "Mykola Matvienko", "5": "Lasha Dvali", "9": "Giorgi Maisuradze", "10": "Otar Kakabadze", "17": "Saba Sazonov", "27": "Irakli Azarov", "31": "Saba Goglichidze", "7": "Jemal Tabidze"}, "fark": 0.01, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303421, "yontem": "değere göre"}, "701|0.0": {"ad": "Oyuncu Gol Atar", "secenekler": {"4": "Artem Dovbyk", "34": "Vladyslav Vanat", "35": "Roman Yaremchuk", "12": "Danylo Sikan", "20": "Igor Krasnopir", "28": "Georges Mikautadze", "39": "Budu Zivzivadze", "8": "Giorgi Guliashvili", "29": "Georgiy Sudakov", "30": "Hennadii Synchuk", "14": "Giorgi Kvernadze", "11": "Oleksiy Kashchuk", "3": "Zuriko Davitashvili", "2": "Giorgi Chakvetadze", "24": "Oleh Ocheretko", "37": "Irakli Yegoian", "23": "Yehor Nazaryna", "33": "Georgiy Tsitaishvili", "21": "Giorgi Moistsrapeshvili", "36": "Nika Gagnidze", "13": "Boris Nikolaevich Krushinsky", "25": "Varfolomieiev Ivan Oleksandrovych", "26": "Eduard Sarapii", "6": "Yehor Yarmoliuk", "16": "Nodari Lominadze", "32": "Mykhavko Taras", "22": "Vitaliy Mykolenko", "19": "Anzor Mekvabishvili", "15": "Luka Lochoshvili", "38": "I. Zabarnyi", "18": "Mykola Matvienko", "1": "Ilya Alekseevich Krupsky", "5": "Lasha Dvali", "17": "Giorgi Maisuradze", "10": "Otar Kakabadze", "27": "Saba Sazonov", "9": "Irakli Azarov", "31": "Saba Goglichidze", "7": "Jemal Tabidze"}, "fark": 0.0173, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303421, "yontem": "değere göre"}, "701|*": {"ad": "Oyuncu Gol Atar", "secenekler": {"4": "Artem Dovbyk", "34": "Vladyslav Vanat", "35": "Roman Yaremchuk", "12": "Danylo Sikan", "20": "Igor Krasnopir", "28": "Georges Mikautadze", "39": "Budu Zivzivadze", "8": "Giorgi Guliashvili", "29": "Georgiy Sudakov", "30": "Hennadii Synchuk", "14": "Giorgi Kvernadze", "11": "Oleksiy Kashchuk", "3": "Zuriko Davitashvili", "2": "Giorgi Chakvetadze", "24": "Oleh Ocheretko", "37": "Irakli Yegoian", "23": "Yehor Nazaryna", "33": "Georgiy Tsitaishvili", "21": "Giorgi Moistsrapeshvili", "36": "Nika Gagnidze", "13": "Boris Nikolaevich Krushinsky", "25": "Varfolomieiev Ivan Oleksandrovych", "26": "Eduard Sarapii", "6": "Yehor Yarmoliuk", "16": "Nodari Lominadze", "32": "Mykhavko Taras", "22": "Vitaliy Mykolenko", "19": "Anzor Mekvabishvili", "15": "Luka Lochoshvili", "38": "I. Zabarnyi", "18": "Mykola Matvienko", "1": "Ilya Alekseevich Krupsky", "5": "Lasha Dvali", "17": "Giorgi Maisuradze", "10": "Otar Kakabadze", "27": "Saba Sazonov", "9": "Irakli Azarov", "31": "Saba Goglichidze", "7": "Jemal Tabidze"}, "fark": 0.0173, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303421, "yontem": "değere göre"}, "716|0.0": {"ad": "Karşılaşma Özel Bahisleri", "secenekler": {"1": "İki Takım da En Az 2 Gol Atar", "2": "Herhangi Bir Oyuncu Hat-Trick Yapar", "3": "Herhangi Bir Oyuncu 2+ Gol Atar", "4": "Maçta Kendi Kalesine Atılan Gol Olur", "5": "Yedekten Giren Oyuncu Gol Atar", "6": "Yedekten Giren Oyuncu Kart Görür", "7": "Ceza Sahası Dışından Gol Atılır", "8": "Kafa Vuruşu İle Gol Atılır", "9": "Herhangi Bir Takım Geriden Gelip Kazanır", "10": "Herhangi Bir Yarı Berabere Biter", "11": "Gol Dakikaları Toplamı En Az 151 Olur", "12": "00:00-05:00 Dakika Aralığında Gol Olur", "13": "00:00-10:00 Dakika Aralığında Gol Olur", "14": "00:00-15:00 Dakika Aralığında Gol Olur", "15": "00:00-20:00 Dakika Aralığında Gol Olur", "16": "00:00-25:00 Dakika Aralığında Gol Olur", "17": "00:00-30:00 Dakika Aralığında Gol Olur", "18": "00:00-05:00 Dk. Ara. Korner Kullanılır", "19": "00:00-10:00 Dk. Ara. Korner Kullanılır", "20": "00:00-15:00 Dk. Ara. 2+Korner Kullanılır", "21": "00:00-15:00 Dk. Ara. Korner Kullanılır", "22": "İki Takım da Her Yarı Korner Kullanır", "23": "İki Takım da Her Yarı 2+ Korner Kullanır", "24": "İki Takım da Her Yarı 3+ Korner Kullanır", "25": "İki Takım da Her Yarı 4+ Korner Kullanır", "26": "İki Takım da En Az 2 Kart Görür", "27": "İki Takım da En Az 3 Kart Görür", "28": "İki Takım da Her Yarıda Kart Görür"}, "fark": 0.0, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303517, "yontem": "sıralı"}, "716|*": {"ad": "Karşılaşma Özel Bahisleri", "secenekler": {"1": "İki Takım da En Az 2 Gol Atar", "2": "Herhangi Bir Oyuncu Hat-Trick Yapar", "3": "Herhangi Bir Oyuncu 2+ Gol Atar", "4": "Maçta Kendi Kalesine Atılan Gol Olur", "5": "Yedekten Giren Oyuncu Gol Atar", "6": "Yedekten Giren Oyuncu Kart Görür", "7": "Ceza Sahası Dışından Gol Atılır", "8": "Kafa Vuruşu İle Gol Atılır", "9": "Herhangi Bir Takım Geriden Gelip Kazanır", "10": "Herhangi Bir Yarı Berabere Biter", "11": "Gol Dakikaları Toplamı En Az 151 Olur", "12": "00:00-05:00 Dakika Aralığında Gol Olur", "13": "00:00-10:00 Dakika Aralığında Gol Olur", "14": "00:00-15:00 Dakika Aralığında Gol Olur", "15": "00:00-20:00 Dakika Aralığında Gol Olur", "16": "00:00-25:00 Dakika Aralığında Gol Olur", "17": "00:00-30:00 Dakika Aralığında Gol Olur", "18": "00:00-05:00 Dk. Ara. Korner Kullanılır", "19": "00:00-10:00 Dk. Ara. Korner Kullanılır", "20": "00:00-15:00 Dk. Ara. 2+Korner Kullanılır", "21": "00:00-15:00 Dk. Ara. Korner Kullanılır", "22": "İki Takım da Her Yarı Korner Kullanır", "23": "İki Takım da Her Yarı 2+ Korner Kullanır", "24": "İki Takım da Her Yarı 3+ Korner Kullanır", "25": "İki Takım da Her Yarı 4+ Korner Kullanır", "26": "İki Takım da En Az 2 Kart Görür", "27": "İki Takım da En Az 3 Kart Görür", "28": "İki Takım da Her Yarıda Kart Görür"}, "fark": 0.0, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303517, "yontem": "sıralı"}, "806|0.0": {"ad": "Takım İsabetli Şut", "secenekler": {"1": "{ev} 5+", "2": "{ev} 6+", "3": "{ev} 7+", "4": "{ev} 8+", "5": "{dep} 5+", "6": "{dep} 6+", "7": "{dep} 7+", "8": "{dep} 8+", "9": "Toplam 10+", "10": "Toplam 11+", "11": "Toplam 12+", "12": "Toplam 13+", "13": "Toplam 14+"}, "fark": 0.0, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303517, "yontem": "sıralı"}, "806|*": {"ad": "Takım İsabetli Şut", "secenekler": {"1": "{ev} 5+", "2": "{ev} 6+", "3": "{ev} 7+", "4": "{ev} 8+", "5": "{dep} 5+", "6": "{dep} 6+", "7": "{dep} 7+", "8": "{dep} 8+", "9": "Toplam 10+", "10": "Toplam 11+", "11": "Toplam 12+", "12": "Toplam 13+", "13": "Toplam 14+"}, "fark": 0.0, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303517, "yontem": "sıralı"}, "805|0.0": {"ad": "Takım Şut", "secenekler": {"1": "{ev} 14+", "2": "{ev} 16+", "3": "{ev} 18+", "4": "{dep} 14+", "5": "{dep} 16+", "6": "{dep} 18+", "7": "Toplam 28+", "8": "Toplam 30+", "9": "Toplam 32+", "10": "Toplam 34+"}, "fark": 0.0, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303517, "yontem": "sıralı"}, "805|*": {"ad": "Takım Şut", "secenekler": {"1": "{ev} 14+", "2": "{ev} 16+", "3": "{ev} 18+", "4": "{dep} 14+", "5": "{dep} 16+", "6": "{dep} 18+", "7": "Toplam 28+", "8": "Toplam 30+", "9": "Toplam 32+", "10": "Toplam 34+"}, "fark": 0.0, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303517, "yontem": "sıralı"}, "808|0.0": {"ad": "Takım Ofsayt", "secenekler": {"1": "{ev} 2+", "2": "{ev} 3+", "3": "{ev} 4+", "4": "{dep} 2+", "5": "{dep} 3+", "6": "{dep} 4+", "7": "Toplam 4+", "8": "Toplam 5+", "9": "Toplam 6+", "10": "Toplam 7+"}, "fark": 0.0, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303517, "yontem": "sıralı"}, "808|*": {"ad": "Takım Ofsayt", "secenekler": {"1": "{ev} 2+", "2": "{ev} 3+", "3": "{ev} 4+", "4": "{dep} 2+", "5": "{dep} 3+", "6": "{dep} 4+", "7": "Toplam 4+", "8": "Toplam 5+", "9": "Toplam 6+", "10": "Toplam 7+"}, "fark": 0.0, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303517, "yontem": "sıralı"}, "803|0.0": {"ad": "Kaleci En Az Kaç Kurtarış Yapar? (Uzt. Dahil)", "secenekler": {"1": "Egil Selvik 3+", "2": "Egil Selvik 4+", "3": "Egil Selvik 5+", "4": "Mads Hermansen 3+", "5": "Mads Hermansen 4+", "6": "Mads Hermansen 5+", "7": "Toplam 6+", "8": "Toplam 7+", "9": "Toplam 8+", "10": "Toplam 9+"}, "fark": 0.0304, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303517, "yontem": "sıralı"}, "803|*": {"ad": "Kaleci En Az Kaç Kurtarış Yapar? (Uzt. Dahil)", "secenekler": {"1": "Egil Selvik 3+", "2": "Egil Selvik 4+", "3": "Egil Selvik 5+", "4": "Mads Hermansen 3+", "5": "Mads Hermansen 4+", "6": "Mads Hermansen 5+", "7": "Toplam 6+", "8": "Toplam 7+", "9": "Toplam 8+", "10": "Toplam 9+"}, "fark": 0.0304, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303517, "yontem": "sıralı"}, "709|0.0": {"ad": "Oyuncu Kart Görür (Uzt. Dahil)", "secenekler": {"1": "Morten Hjulmand", "2": "Joakim Maehle", "10": "Alexander Sorloth", "11": "Erling Haaland", "3": "Pierre Emile Hojbjerg", "12": "Kristoffer Ajer", "13": "Sander Berge", "4": "Alexander Bah", "6": "Oliver Provstgaard", "5": "Mikkel Damsgaard", "14": "Oscar Bobb", "15": "Martin Odegaard", "7": "Gustav Isaksen", "8": "Rasmus Hojlund", "16": "Antonio Nusa", "9": "Mohamed Daramy"}, "fark": 0.0, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303517, "yontem": "değere göre"}, "709|*": {"ad": "Oyuncu Kart Görür (Uzt. Dahil)", "secenekler": {"1": "Morten Hjulmand", "2": "Joakim Maehle", "10": "Alexander Sorloth", "11": "Erling Haaland", "3": "Pierre Emile Hojbjerg", "12": "Kristoffer Ajer", "13": "Sander Berge", "4": "Alexander Bah", "6": "Oliver Provstgaard", "5": "Mikkel Damsgaard", "14": "Oscar Bobb", "15": "Martin Odegaard", "7": "Gustav Isaksen", "8": "Rasmus Hojlund", "16": "Antonio Nusa", "9": "Mohamed Daramy"}, "fark": 0.0, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303517, "yontem": "değere göre"}, "712|0.0": {"ad": "Oyuncu Asist Yapar (Uzt. Dahil)", "secenekler": {"10": "Martin Odegaard 1+", "2": "Mikkel Damsgaard 1+", "11": "Antonio Nusa 1+", "5": "Joakim Maehle 1+", "3": "Gustav Isaksen 1+", "12": "Andreas Schjelderup 1+", "9": "Alexander Sorloth 1+", "6": "Alexander Bah 1+", "4": "Mohamed Daramy 1+", "8": "Erling Haaland 1+", "14": "Patrick Berg 1+", "7": "Rasmus Hojlund 1+", "13": "Morten Hjulmand 1+", "1": "Oscar Bobb 1+"}, "fark": 0.0, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303517, "yontem": "değere göre"}, "712|*": {"ad": "Oyuncu Asist Yapar (Uzt. Dahil)", "secenekler": {"10": "Martin Odegaard 1+", "2": "Mikkel Damsgaard 1+", "11": "Antonio Nusa 1+", "5": "Joakim Maehle 1+", "3": "Gustav Isaksen 1+", "12": "Andreas Schjelderup 1+", "9": "Alexander Sorloth 1+", "6": "Alexander Bah 1+", "4": "Mohamed Daramy 1+", "8": "Erling Haaland 1+", "14": "Patrick Berg 1+", "7": "Rasmus Hojlund 1+", "13": "Morten Hjulmand 1+", "1": "Oscar Bobb 1+"}, "fark": 0.0, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303517, "yontem": "değere göre"}, "765|0.0": {"ad": "Oyuncu Gol Atar veya Asist Yapar (Uzt. Dahil)", "secenekler": {"7": "Erling Haaland", "1": "Rasmus Hojlund", "2": "Mikkel Damsgaard", "3": "Gustav Isaksen", "8": "Alexander Sorloth", "9": "Martin Odegaard", "4": "Mohamed Daramy", "10": "Antonio Nusa", "11": "Oscar Bobb", "5": "Joakim Maehle", "12": "Patrick Berg", "6": "Morten Hjulmand", "13": "Sander Berge"}, "fark": 0.0, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303517, "yontem": "değere göre"}, "765|*": {"ad": "Oyuncu Gol Atar veya Asist Yapar (Uzt. Dahil)", "secenekler": {"7": "Erling Haaland", "1": "Rasmus Hojlund", "2": "Mikkel Damsgaard", "3": "Gustav Isaksen", "8": "Alexander Sorloth", "9": "Martin Odegaard", "4": "Mohamed Daramy", "10": "Antonio Nusa", "11": "Oscar Bobb", "5": "Joakim Maehle", "12": "Patrick Berg", "6": "Morten Hjulmand", "13": "Sander Berge"}, "fark": 0.0, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303517, "yontem": "değere göre"}, "740|0.0": {"ad": "Oyuncu Şut Çeker (Uzt. Dahil)", "secenekler": {"28": "Alexander Sorloth 2+", "1": "Mohamed Daramy 2+", "13": "Morten Hjulmand 1+", "22": "Antonio Nusa 2+", "10": "Pierre Emile Hojbjerg 1+", "19": "Erling Haaland 4+", "7": "Gustav Isaksen 3+", "4": "Rasmus Hojlund 3+", "16": "Oscar Bobb 2+", "25": "Martin Odegaard 2+", "29": "Alexander Sorloth 3+", "20": "Erling Haaland 5+", "2": "Mohamed Daramy 3+", "23": "Antonio Nusa 3+", "8": "Gustav Isaksen 4+", "5": "Rasmus Hojlund 4+", "14": "Morten Hjulmand 2+", "17": "Oscar Bobb 3+", "21": "Erling Haaland 6+", "11": "Pierre Emile Hojbjerg 2+", "30": "Alexander Sorloth 4+", "26": "Martin Odegaard 3+", "3": "Mohamed Daramy 4+", "24": "Gustav Isaksen 5+", "9": "Antonio Nusa 4+", "6": "Rasmus Hojlund 5+", "18": "Oscar Bobb 4+", "15": "Morten Hjulmand 3+", "27": "Martin Odegaard 4+", "12": "Pierre Emile Hojbjerg 3+"}, "fark": 0.0, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303517, "yontem": "değere göre"}, "740|*": {"ad": "Oyuncu Şut Çeker (Uzt. Dahil)", "secenekler": {"28": "Alexander Sorloth 2+", "1": "Mohamed Daramy 2+", "13": "Morten Hjulmand 1+", "22": "Antonio Nusa 2+", "10": "Pierre Emile Hojbjerg 1+", "19": "Erling Haaland 4+", "7": "Gustav Isaksen 3+", "4": "Rasmus Hojlund 3+", "16": "Oscar Bobb 2+", "25": "Martin Odegaard 2+", "29": "Alexander Sorloth 3+", "20": "Erling Haaland 5+", "2": "Mohamed Daramy 3+", "23": "Antonio Nusa 3+", "8": "Gustav Isaksen 4+", "5": "Rasmus Hojlund 4+", "14": "Morten Hjulmand 2+", "17": "Oscar Bobb 3+", "21": "Erling Haaland 6+", "11": "Pierre Emile Hojbjerg 2+", "30": "Alexander Sorloth 4+", "26": "Martin Odegaard 3+", "3": "Mohamed Daramy 4+", "24": "Gustav Isaksen 5+", "9": "Antonio Nusa 4+", "6": "Rasmus Hojlund 5+", "18": "Oscar Bobb 4+", "15": "Morten Hjulmand 3+", "27": "Martin Odegaard 4+", "12": "Pierre Emile Hojbjerg 3+"}, "fark": 0.0, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303517, "yontem": "değere göre"}, "714|0.0": {"ad": "Oyuncu Kaleyi Bulan Şut Çeker (Uzt. Dahil)", "secenekler": {"7": "Gustav Isaksen 1+", "4": "Rasmus Hojlund 1+", "1": "Mohammed Daramy 1+", "11": "Mikkel Damsgaard 1+", "14": "Erling Haaland 2+", "20": "Antonio Nusa 1+", "17": "Oscar Bobb 1+", "26": "Thelo Aasgaard 1+", "23": "Martin Odegaard 1+", "8": "Gustav Isaksen 2+", "15": "Erling Haaland 3+", "5": "Rasmus Hojlund 2+", "10": "Morten Hjulmand 1+", "25": "Sander Berge 1+", "12": "Mohammed Daramy 2+", "2": "Mikkel Damsgaard 2+", "21": "Antonio Nusa 2+", "18": "Oscar Bobb 2+", "16": "Erling Haaland 4+", "9": "Gustav Isaksen 3+", "6": "Rasmus Hojlund 3+", "24": "Martin Odegaard 2+", "13": "Mohammed Daramy 3+", "3": "Mikkel Damsgaard 3+", "22": "Antonio Nusa 3+", "19": "Oscar Bobb 3+"}, "fark": 0.0, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303517, "yontem": "değere göre"}, "714|*": {"ad": "Oyuncu Kaleyi Bulan Şut Çeker (Uzt. Dahil)", "secenekler": {"7": "Gustav Isaksen 1+", "4": "Rasmus Hojlund 1+", "1": "Mohammed Daramy 1+", "11": "Mikkel Damsgaard 1+", "14": "Erling Haaland 2+", "20": "Antonio Nusa 1+", "17": "Oscar Bobb 1+", "26": "Thelo Aasgaard 1+", "23": "Martin Odegaard 1+", "8": "Gustav Isaksen 2+", "15": "Erling Haaland 3+", "5": "Rasmus Hojlund 2+", "10": "Morten Hjulmand 1+", "25": "Sander Berge 1+", "12": "Mohammed Daramy 2+", "2": "Mikkel Damsgaard 2+", "21": "Antonio Nusa 2+", "18": "Oscar Bobb 2+", "16": "Erling Haaland 4+", "9": "Gustav Isaksen 3+", "6": "Rasmus Hojlund 3+", "24": "Martin Odegaard 2+", "13": "Mohammed Daramy 3+", "3": "Mikkel Damsgaard 3+", "22": "Antonio Nusa 3+", "19": "Oscar Bobb 3+"}, "fark": 0.0, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303517, "yontem": "değere göre"}, "807|0.0": {"ad": "Takım Faul", "secenekler": {"1": "{ev} 11+", "2": "{ev} 13+", "3": "{ev} 15+", "10": "İY {ev} 5+", "11": "İY {ev} 7+", "12": "İY {dep} 6+", "13": "İY {dep} 8+", "14": "İY Toplam 11+", "15": "İY Toplam 13+", "16": "İY Toplam 15+", "4": "{dep} 12+", "5": "{dep} 14+", "6": "{dep} 16+", "7": "Toplam 23+", "8": "Toplam 25+", "9": "Toplam 27+"}, "fark": 0.0, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303517, "yontem": "değere göre"}, "807|*": {"ad": "Takım Faul", "secenekler": {"1": "{ev} 11+", "2": "{ev} 13+", "3": "{ev} 15+", "10": "İY {ev} 5+", "11": "İY {ev} 7+", "12": "İY {dep} 6+", "13": "İY {dep} 8+", "14": "İY Toplam 11+", "15": "İY Toplam 13+", "16": "İY Toplam 15+", "4": "{dep} 12+", "5": "{dep} 14+", "6": "{dep} 16+", "7": "Toplam 23+", "8": "Toplam 25+", "9": "Toplam 27+"}, "fark": 0.0, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303517, "yontem": "değere göre"}, "871|0.0": {"ad": "Takım Kale Vuruşu", "secenekler": {"2": "{ev} 10+", "3": "{ev} 12+", "1": "{ev} 8+", "11": "İY {ev} 5+", "12": "İY {ev} 6+", "13": "İY {ev} 7+", "14": "İY {dep} 5+", "15": "İY {dep} 6+", "16": "İY {dep} 7+", "18": "İY Toplam 10+", "19": "İY Toplam 11+", "20": "İY Toplam 12+", "21": "İY Toplam 13+", "17": "İY Toplam 9+", "5": "{dep} 10+", "6": "{dep} 12+", "4": "{dep} 8+", "7": "Toplam 16+", "8": "Toplam 18+", "9": "Toplam 20+", "10": "Toplam 22+"}, "fark": 0.0, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303517, "yontem": "değere göre"}, "871|*": {"ad": "Takım Kale Vuruşu", "secenekler": {"2": "{ev} 10+", "3": "{ev} 12+", "1": "{ev} 8+", "11": "İY {ev} 5+", "12": "İY {ev} 6+", "13": "İY {ev} 7+", "14": "İY {dep} 5+", "15": "İY {dep} 6+", "16": "İY {dep} 7+", "18": "İY Toplam 10+", "19": "İY Toplam 11+", "20": "İY Toplam 12+", "21": "İY Toplam 13+", "17": "İY Toplam 9+", "5": "{dep} 10+", "6": "{dep} 12+", "4": "{dep} 8+", "7": "Toplam 16+", "8": "Toplam 18+", "9": "Toplam 20+", "10": "Toplam 22+"}, "fark": 0.0, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303517, "yontem": "değere göre"}, "872|0.0": {"ad": "Takım Taç Atışı", "secenekler": {"1": "{ev} 16+", "2": "{ev} 18+", "3": "{ev} 20+", "10": "İY {ev} 9+", "11": "İY {dep} 9+", "12": "İY Toplam 16+", "13": "İY Toplam 18+", "4": "{dep} 15+", "5": "{dep} 17+", "6": "{dep} 19+", "7": "Toplam 31+", "8": "Toplam 33+", "9": "Toplam 35+"}, "fark": 0.0, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303517, "yontem": "değere göre"}, "872|*": {"ad": "Takım Taç Atışı", "secenekler": {"1": "{ev} 16+", "2": "{ev} 18+", "3": "{ev} 20+", "10": "İY {ev} 9+", "11": "İY {dep} 9+", "12": "İY Toplam 16+", "13": "İY Toplam 18+", "4": "{dep} 15+", "5": "{dep} 17+", "6": "{dep} 19+", "7": "Toplam 31+", "8": "Toplam 33+", "9": "Toplam 35+"}, "fark": 0.0, "supheli": false, "maca_ozel": true, "ogrenilen_mac": 3303517, "yontem": "değere göre"}}''')

import os
SOZLUK_DOSYASI = "market_sozlugu_v3.json"   # eski market_sozlugu*.json dosyalarını KULLANMA
sozluk = json.load(open(SOZLUK_DOSYASI, encoding="utf-8")) if os.path.exists(SOZLUK_DOSYASI) else dict(HAZIR_SOZLUK)
print("Bilinen market kodu:", len([k for k in sozluk if not k.endswith('|*')]))

ornekler = []
for takim, metin in REFERANSLAR:
    bulunan = maci_bul(tum_maclar, takim)
    if not bulunan:
        print(f"[{takim}] bültende yok (maç başlamış olabilir); atlandı.")
        continue
    mac = bulunan[0]
    referans = referans_ayristir(metin)
    print(f"[{mac.get('HN')} - {mac.get('AN')}] metinde {len(referans)} market, bültende {len(mac.get('MA') or [])} market")
    ornekler.append((mac, referans))

if ornekler:
    yeni, rapor, atanmayan = sozluk_ogren_ortak(ornekler, esik=0.12)
    eklenen = 0
    for k, v in yeni.items():
        if k not in sozluk:
            sozluk[k] = v; eklenen += 1
        elif not k.endswith("|*") and sozluk[k]["ad"] != v["ad"]:
            print(f"   ! Kod {k}: sözlükte '{sozluk[k]['ad']}', yeni örnek '{v['ad']}' diyor → sözlükteki tutuldu")
    print(f"Yeni eklenen kod: {eklenen}; eşleşmeyen isimler: {atanmayan}")
    if not rapor.empty:
        display(rapor.sort_values("ortanca_fark_%", ascending=False).head(15))

json.dump(sozluk, open(SOZLUK_DOSYASI, "w", encoding="utf-8"), ensure_ascii=False)
print("Toplam bilinen market kodu:", len([k for k in sozluk if not k.endswith('|*')]))
try:
    from google.colab import files
    if ornekler:
        files.download(SOZLUK_DOSYASI)
except ImportError:
    pass

In [ ]:
# 4) Bugün 00:00 – yarın 03:00 (TR saati) arasındaki maçlar, isimli
bas, bit = zaman_penceresi()
print("Pencere:", bas.strftime("%d.%m %H:%M"), "→", bit.strftime("%d.%m %H:%M"))
maclar = pencereye_gore_filtrele(tum_maclar, bas, bit)
if not maclar:
    print("Bu saatte başlamamış maç kalmamış (bülten yalnızca başlamamış maçları içerir). Canlı maçlar için 6. hücreye bak.")
else:
    df = isim_ekle(duzlestir(maclar), sozluk)
    acik = df.dropna(subset=["oran"])
    isimsiz = acik["market_adi"].str.startswith("market ").mean()
    print(f"{len(maclar)} maç, {len(acik)} açık oran; isimlendirilemeyen oran oranı: %{isimsiz*100:.0f}")
    print("Spor koduna göre isimli oran oranı:",
          (~acik["market_adi"].str.startswith("market ")).groupby(acik["spor_kodu"]).mean().round(2).to_dict())

    ozet = (df.groupby(["kod", "karsilasma", "spor_kodu", "lig", "tarih", "saat", "ev", "deplasman"], dropna=False)
              .agg(market_sayisi=("market_id", "nunique"), acik_oran=("oran", "count"))
              .reset_index().sort_values(["tarih", "saat"]))
    genis = genis_tablo(df)
    uzun = df[["kod", "karsilasma", "spor_kodu", "lig", "tarih", "saat", "ev", "deplasman", "market_adi", "secenek_adi", "oran",
               "market_id", "ozel_deger", "secenek_no", "mac_mbs"]]

    dosya = f"nesine_{bas.strftime('%Y%m%d')}_0300.xlsx"
    with pd.ExcelWriter(dosya) as w:
        ozet.to_excel(w, sheet_name="ozet", index=False)
        uzun.to_excel(w, sheet_name="tum_oranlar", index=False)
        genis.to_excel(w, sheet_name="maclar_genis", index=False)
    try:
        from google.colab import files
        files.download(dosya)
    except ImportError:
        print("Kaydedildi:", dosya)
    ozet.head(20)

In [ ]:
# 5) Tek maça bak (takım adından bir parça ya da kod=...)
if "df" in globals() and not df.empty:
    mac_goster(df, takim="Galatasaray")

In [ ]:
# 6) CANLI maçlar (şu an oynananlar) – ayrı bir Excel olarak iner
# Canlı oranlar başka bir kaynaktan geliyor: canlı bahis sayfası Colab içinde gerçek bir tarayıcıyla açılır,
# sayfanın arka planda aldığı veriler yakalanır. İlk çalıştırmada kurulum 1-2 dakika sürer.
CANLI = True
CANLI_SAYFA = "https://www.nesine.com/iddaa/canli-iddaa-canli-bahis"   # açılmazsa tarayıcındaki canlı bahis adresini yapıştır

import asyncio, json, re

import asyncio, json, re

async def canli_bulten_yakala(sayfa_url, bekle_sn=25):
    """Canlı bahis sayfasını gerçek bir tarayıcıda açar; sayfanın arka planda aldığı bütün JSON yanıtlarını
    ve WebSocket mesajlarını toplar. İçinde maç listesi (MA alanlı) olan en büyük veriyi döndürür."""
    from playwright.async_api import async_playwright
    govdeler = []                                   # (kaynak, metin)

    async def yanit(resp):
        try:
            if "json" in resp.headers.get("content-type", "") and "nesine" in resp.url:
                govdeler.append((resp.url, await resp.text()))
        except Exception:
            pass

    def ws(w):
        def al(veri):
            if isinstance(veri, str) and len(govdeler) < 3000:
                govdeler.append(("ws:" + w.url, veri))
        w.on("framereceived", al)

    async with async_playwright() as p:
        tarayici = await p.chromium.launch(headless=True, args=["--no-sandbox"])
        sayfa = await tarayici.new_page(locale="tr-TR", user_agent=HEADERS["User-Agent"])
        sayfa.on("response", yanit)
        sayfa.on("websocket", ws)
        await sayfa.goto(sayfa_url, wait_until="domcontentloaded", timeout=60000)
        await asyncio.sleep(bekle_sn)
        await tarayici.close()

    adaylar = []
    for kaynak, metin in govdeler:
        for parca in metin.split("\x1e"):           # SignalR mesaj ayırıcısı
            parca = parca.strip()
            if not parca or parca[0] not in "[{":
                continue
            try:
                veri = json.loads(parca)
            except ValueError:
                continue
            maclar = mac_listesini_bul(veri)
            if maclar:
                adaylar.append((len(maclar), kaynak, maclar))
    adaylar.sort(key=lambda x: -x[0])
    print(f"Yakalanan yanıt/mesaj: {len(govdeler)}, içinde maç listesi olan: {len(adaylar)}")
    for n, kaynak, _ in adaylar[:5]:
        print(f"   {n} maç  ←  {kaynak[:120]}")
    return adaylar

def canli_duzlestir(maclar):
    """Canlı maçlar: oranlar + maçın bültende gelen diğer bütün basit alanları (skor, dakika vb. olabilir)."""
    df = duzlestir(maclar)
    bilinen = {"C", "TYPE", "LN", "D", "T", "HN", "AN", "MBS", "MA"}
    ek = []
    for m in maclar:
        ek.append({"kod": m.get("C"), **{f"alan_{k}": v for k, v in m.items()
                                          if k not in bilinen and isinstance(v, (str, int, float, bool))}})
    ek = pd.DataFrame(ek).drop_duplicates("kod")
    return df.merge(ek, on="kod", how="left") if not ek.empty else df


def canli_maclari_sec(adaylar, onceki_maclar):
    """Yakalanan bütün listeleri birleştirir (aynı maç birden çok yerde geçiyorsa en çok marketli sürümü alınır),
    maç öncesi bültende olan maçları çıkarır. Kalanlar = şu an canlı (ya da maç öncesi listede olmayan) maçlar."""
    birlesik = {}
    for _, kaynak, maclar in adaylar:
        for m in maclar:
            k = m.get("C")
            if k is None:
                continue
            if k not in birlesik or len(m.get("MA") or []) > len(birlesik[k].get("MA") or []):
                birlesik[k] = m
    onceki = {m.get("C") for m in onceki_maclar}
    canli = [m for k, m in birlesik.items() if k not in onceki]
    print(f"Yakalanan toplam farklı maç: {len(birlesik)}, maç öncesi bültende olanlar çıkarıldı → canlı: {len(canli)}")
    return canli


if CANLI:
    !pip -q install playwright
    !playwright install --with-deps chromium > /dev/null 2>&1 && echo 'Chromium hazır'
    adaylar = await canli_bulten_yakala(CANLI_SAYFA, bekle_sn=25)
    if not adaylar:
        print("Canlı maç verisi yakalanamadı. Sayfa adresini kontrol et ya da bu hücrenin çıktısını Claude'a gönder.")
    else:
        canli_maclar = canli_maclari_sec(adaylar, tum_maclar)
    if adaylar and not canli_maclar:
        print("Yakalanan listelerde maç öncesi bültende olmayan maç yok; canlı veri farklı bir yoldan geliyor olabilir. Bu çıktıyı Claude'a gönder.")
    elif adaylar:
        cdf = isim_ekle(canli_duzlestir(canli_maclar), sozluk)
        acik_c = cdf.dropna(subset=["oran"])
        print(f"{cdf['kod'].nunique()} canlı maç, {len(acik_c)} açık oran; isimli oran oranı: "
              f"%{(~acik_c['market_adi'].str.startswith('market ')).mean()*100:.0f}")
        alanlar = [c for c in cdf.columns if c.startswith("alan_")]
        if alanlar:
            print("Maçla gelen diğer alanlar (skor/dakika bunlardan biri olabilir):")
            display(cdf.drop_duplicates("kod")[["ev", "deplasman"] + alanlar].head(10))
        from datetime import datetime
        dosya_c = f"nesine_canli_{datetime.now(TR).strftime('%Y%m%d_%H%M')}.xlsx"
        with pd.ExcelWriter(dosya_c) as w:
            cdf.to_excel(w, sheet_name="tum_oranlar", index=False)
            genis_tablo(cdf).to_excel(w, sheet_name="maclar_genis", index=False)
        try:
            from google.colab import files
            files.download(dosya_c)
        except ImportError:
            print("Kaydedildi:", dosya_c)

## Plan B – sadece 2. hücre hata verirse: aşağıda `PLAN_B = True` yapıp bu hücreyi çalıştır

In [ ]:
PLAN_B = False   # Normalde kapalı. Bülten çekilemezse True yap ve sadece bu hücreyi çalıştır.

import asyncio, json, time
from playwright.async_api import async_playwright

async def canli_kesif(sayfa_url, bekle_sn=40, alan_filtre="nesine"):
    """Canlı sayfayı gerçek bir tarayıcıda açar, sayfanın arka planda yaptığı
    JSON isteklerini ve WebSocket mesajlarını kaydeder."""
    istekler, ws_kayit = [], []

    async def yanit_yakala(resp):
        try:
            ct = resp.headers.get("content-type", "")
            if alan_filtre in resp.url and "json" in ct:
                govde = await resp.text()
                istekler.append({"url": resp.url, "durum": resp.status,
                                 "boyut_kb": round(len(govde) / 1024, 1),
                                 "ornek": govde[:300]})
        except Exception:
            pass

    def ws_yakala(ws):
        kayit = {"url": ws.url, "mesajlar": []}
        ws_kayit.append(kayit)
        def al(veri):
            if len(kayit["mesajlar"]) < 200:
                kayit["mesajlar"].append(veri if isinstance(veri, str) else f"<binary {len(veri)} bayt>")
        ws.on("framereceived", al)

    async with async_playwright() as p:
        tarayici = await p.chromium.launch(headless=True, args=["--no-sandbox"])
        sayfa = await tarayici.new_page(locale="tr-TR", user_agent=
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0 Safari/537.36")
        sayfa.on("response", yanit_yakala)
        sayfa.on("websocket", ws_yakala)
        await sayfa.goto(sayfa_url, wait_until="domcontentloaded", timeout=60000)
        print("Sayfa açıldı:", await sayfa.title())
        await asyncio.sleep(bekle_sn)
        await tarayici.close()
    return istekler, ws_kayit


if PLAN_B:
    !pip -q install playwright
    !playwright install --with-deps chromium > /dev/null 2>&1 && echo 'Chromium hazır'
    SAYFA = "https://www.nesine.com/iddaa?et=1&le=2&gt=Pop%C3%BCler&ocg=MS-2%2C5&q="
    istekler, ws_kayit = await canli_kesif(SAYFA, bekle_sn=30)
    kesif_df = pd.DataFrame(istekler)
    pd.set_option("display.max_colwidth", 140)
    if not kesif_df.empty:
        display(kesif_df.sort_values("boyut_kb", ascending=False)[["url", "durum", "boyut_kb"]].head(25))
    else:
        print("JSON isteği yakalanmadı; sayfa erişimi engellenmiş olabilir.")
    with open("kesif.json", "w", encoding="utf-8") as fh:
        json.dump({"istekler": istekler, "websocket": ws_kayit}, fh, ensure_ascii=False, indent=1)
else:
    print("Plan B kapalı (gerek yok).")